# Credit Card Fraud Analysis — Medallion + Star Schema

**Architecture:** `KAGGLE_DB.BRONZE` → `KAGGLE_DB.SILVER` → `KAGGLE_DB.GOLD`

| Layer | Table | Purpose |
|-------|-------|--------|
| Bronze | `RAW_TRANSACTIONS` | Raw data copy (preserves original, no data loss) |
| Silver | `ENRICHED_TRANSACTIONS` | Cleansed + derived columns (age, distance, time buckets) |
| Gold | **Star Schema** | Fact & Dimension tables for analytics |

### Gold Layer — Star Schema
```
              DIM_DATE
                 |
 DIM_CUSTOMER -- FACT_TRANSACTIONS -- DIM_MERCHANT
                 |
             DIM_LOCATION
```

| Table | Type | Description |
|-------|------|-------------|
| `FACT_TRANSACTIONS` | Fact | Transaction measures (amt, distance, is_fraud) + FK to dimensions |
| `DIM_CUSTOMER` | Dimension | Customer profile (CC_NUM, name, gender, DOB, age, job) |
| `DIM_MERCHANT` | Dimension | Merchant info (name, category, lat/long) |
| `DIM_LOCATION` | Dimension | Geography (city, state, zip, lat/long, city_pop) |
| `DIM_DATE` | Dimension | Calendar (date, year, month, day, quarter, week) |

In [ ]:
%%sql -r overview
SELECT 
    COUNT(*) AS TOTAL_TRANSACTIONS,
    SUM(IS_FRAUD) AS FRAUD_CASES,
    ROUND(SUM(IS_FRAUD) * 100.0 / COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(SUM(CASE WHEN IS_FRAUD = 1 THEN AMT ELSE 0 END), 2) AS TOTAL_FRAUD_LOSS,
    ROUND(AVG(CASE WHEN IS_FRAUD = 1 THEN AMT END), 2) AS AVG_FRAUD_AMOUNT,
    ROUND(AVG(CASE WHEN IS_FRAUD = 0 THEN AMT END), 2) AS AVG_LEGIT_AMOUNT
FROM KAGGLE_DB.GOLD.FACT_TRANSACTIONS

---
## 1. Fraud by Category

In [ ]:
%%sql -r fraud_by_category
SELECT 
    m.CATEGORY,
    COUNT(*) AS TOTAL_TRANSACTIONS,
    SUM(f.IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(f.IS_FRAUD) * 100.0 / COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(AVG(CASE WHEN f.IS_FRAUD = 1 THEN f.AMT END), 2) AS AVG_FRAUD_AMOUNT
FROM KAGGLE_DB.GOLD.FACT_TRANSACTIONS f
JOIN KAGGLE_DB.GOLD.DIM_MERCHANT m ON f.MERCHANT_SK = m.MERCHANT_SK
GROUP BY m.CATEGORY
ORDER BY FRAUD_RATE_PCT DESC

---
## 2. Fraud by Local Hour (Timezone-Adjusted)
Original timestamps are in UTC. We map each state to its US timezone and convert to local time for accurate analysis.

In [ ]:
%%sql -r fraud_by_hour
SELECT 
    f.TRANSACTION_HOUR AS UTC_HOUR,
    s.LOCAL_HOUR,
    s.TIMEZONE,
    COUNT(*) AS TOTAL_TRANSACTIONS,
    SUM(f.IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(f.IS_FRAUD) * 100.0 / COUNT(*), 2) AS FRAUD_RATE_PCT
FROM KAGGLE_DB.GOLD.FACT_TRANSACTIONS f
JOIN KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS s ON f.TRANSACTION_ID = s.ID
GROUP BY f.TRANSACTION_HOUR, s.LOCAL_HOUR, s.TIMEZONE
ORDER BY FRAUD_RATE_PCT DESC
LIMIT 20

---
## 3. Fraud by State (Top 10)

In [ ]:
%%sql -r fraud_by_state
SELECT 
    l.STATE,
    COUNT(*) AS TOTAL_TRANSACTIONS,
    SUM(f.IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(f.IS_FRAUD) * 100.0 / COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(SUM(CASE WHEN f.IS_FRAUD = 1 THEN f.AMT ELSE 0 END), 2) AS FRAUD_LOSS
FROM KAGGLE_DB.GOLD.FACT_TRANSACTIONS f
JOIN KAGGLE_DB.GOLD.DIM_LOCATION l ON f.LOCATION_SK = l.LOCATION_SK
GROUP BY l.STATE
ORDER BY FRAUD_COUNT DESC
LIMIT 10

---
## 4. Fraud by Age Group & Gender

In [ ]:
%%sql -r fraud_by_demographics
SELECT 
    c.GENDER,
    CASE 
        WHEN c.AGE < 25 THEN '18-24'
        WHEN c.AGE < 35 THEN '25-34'
        WHEN c.AGE < 45 THEN '35-44'
        WHEN c.AGE < 55 THEN '45-54'
        WHEN c.AGE < 65 THEN '55-64'
        ELSE '65+'
    END AS AGE_GROUP,
    COUNT(*) AS TOTAL_TRANSACTIONS,
    SUM(f.IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(f.IS_FRAUD) * 100.0 / COUNT(*), 2) AS FRAUD_RATE_PCT
FROM KAGGLE_DB.GOLD.FACT_TRANSACTIONS f
JOIN KAGGLE_DB.GOLD.DIM_CUSTOMER c ON f.CUSTOMER_SK = c.CUSTOMER_SK
GROUP BY c.GENDER, AGE_GROUP
ORDER BY FRAUD_RATE_PCT DESC

---
## 5. Fraud by Transaction Amount

In [ ]:
%%sql -r fraud_by_amount
SELECT 
    f.AMOUNT_BUCKET,
    COUNT(*) AS TOTAL_TRANSACTIONS,
    SUM(f.IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(f.IS_FRAUD) * 100.0 / COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(AVG(f.DISTANCE_KM), 2) AS AVG_DISTANCE_KM
FROM KAGGLE_DB.GOLD.FACT_TRANSACTIONS f
GROUP BY f.AMOUNT_BUCKET
ORDER BY FRAUD_RATE_PCT DESC

---
## 6. Monthly Trend

In [ ]:
%%sql -r fraud_by_month
SELECT 
    d.YEAR,
    d.MONTH_NAME,
    d.MONTH,
    COUNT(*) AS TOTAL_TRANSACTIONS,
    SUM(f.IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(f.IS_FRAUD) * 100.0 / COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(SUM(CASE WHEN f.IS_FRAUD = 1 THEN f.AMT ELSE 0 END), 2) AS FRAUD_LOSS
FROM KAGGLE_DB.GOLD.FACT_TRANSACTIONS f
JOIN KAGGLE_DB.GOLD.DIM_DATE d ON f.DATE_SK = d.DATE_SK
GROUP BY d.YEAR, d.MONTH_NAME, d.MONTH
ORDER BY d.YEAR, d.MONTH

---
## 7. Top 10 Merchants by Fraud

In [ ]:
%%sql -r fraud_by_merchant
SELECT 
    m.MERCHANT,
    m.CATEGORY,
    COUNT(*) AS TOTAL_TRANSACTIONS,
    SUM(f.IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(f.IS_FRAUD) * 100.0 / COUNT(*), 2) AS FRAUD_RATE_PCT
FROM KAGGLE_DB.GOLD.FACT_TRANSACTIONS f
JOIN KAGGLE_DB.GOLD.DIM_MERCHANT m ON f.MERCHANT_SK = m.MERCHANT_SK
WHERE f.IS_FRAUD = 1
GROUP BY m.MERCHANT, m.CATEGORY
ORDER BY FRAUD_COUNT DESC
LIMIT 10

---
## 8. Day of Week Analysis

In [ ]:
%%sql -r fraud_by_day
SELECT 
    d.DAY_NAME,
    COUNT(*) AS TOTAL_TRANSACTIONS,
    SUM(f.IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(f.IS_FRAUD) * 100.0 / COUNT(*), 2) AS FRAUD_RATE_PCT
FROM KAGGLE_DB.GOLD.FACT_TRANSACTIONS f
JOIN KAGGLE_DB.GOLD.DIM_DATE d ON f.DATE_SK = d.DATE_SK
GROUP BY d.DAY_NAME
ORDER BY FRAUD_RATE_PCT DESC

---
## Power BI Star Schema (Gold Layer)

Optimized for **DirectQuery / Import** with pre-built dimensions — no DAX calculations needed for slicing.

```
                    DIM_DATE (194)
                       |
  DIM_CUSTOMER (925)   |   DIM_MERCHANT (555K)
         \             |            /
          \            |           /
   DIM_AGE_BUCKET (7)--FACT_TRANSACTIONS (555K)--DIM_AMOUNT_BUCKET (5)
          /            |           \
         /             |            \
  DIM_TIMEZONE (6)     |   DIM_TIME_SLOT (24)
                       |
                 DIM_RISK_BAND (5)
                       |
                 DIM_LOCATION (912)
```

| Dimension | Rows | Use as Power BI Slicer |
|-----------|------|------------------------|
| `DIM_AGE_BUCKET` | 7 | Age Group, Generation |
| `DIM_AMOUNT_BUCKET` | 5 | Transaction Size (Micro→Very Large) |
| `DIM_TIME_SLOT` | 24 | Hour, Time of Day, Business Hours |
| `DIM_RISK_BAND` | 5 | Risk Level, Action Required |
| `DIM_TIMEZONE` | 6 | Region (Eastern/Central/Mountain/Pacific) |
| `DIM_DATE` | 194 | Year, Month, Quarter, Day of Week |
| `DIM_CUSTOMER` | 925 | Gender, Job, Age |
| `DIM_LOCATION` | 912 | State, City, Population |
| `DIM_MERCHANT` | 555K | Merchant Name, Category |

### Power BI Connection
- **Server:** `id84194.snowflakecomputing.com`
- **Warehouse:** `COMPUTE_WH`
- **Database:** `KAGGLE_DB`
- **Schema:** `GOLD`
- Import all `DIM_*` tables + `FACT_TRANSACTIONS`
- Create relationships on `*_SK` columns
- All slicing is pre-computed — no complex DAX needed

---
## Pipeline Architecture: Streams, Tasks & Stored Procedures

```
[New Data] → BRONZE.RAW_TRANSACTIONS
                    |
             STM_RAW_TRANSACTIONS (Stream)
                    |
         TASK_BRONZE_TO_SILVER (every 5 min, when stream has data)
                    |
             SP_BRONZE_TO_SILVER() → MERGE into SILVER.ENRICHED_TRANSACTIONS
                    |
             STM_ENRICHED_TRANSACTIONS (Stream)
                    |
         TASK_SILVER_TO_GOLD (chained after parent)
                    |
             SP_SILVER_TO_GOLD() → MERGE into GOLD dimensions + INSERT facts
```

**Error Handling:** Both procedures log errors to `BRONZE.PIPELINE_ERROR_LOG` with:
- Procedure name, error code, message, SQL state, stack trace, timestamp
- Then re-raise the exception so the task shows as failed

**Objects Created:**
| Object | Type | Schema |
|--------|------|--------|
| `STM_RAW_TRANSACTIONS` | Stream | BRONZE |
| `STM_ENRICHED_TRANSACTIONS` | Stream | SILVER |
| `PIPELINE_ERROR_LOG` | Table | BRONZE |
| `SP_BRONZE_TO_SILVER` | Procedure | SILVER |
| `SP_SILVER_TO_GOLD` | Procedure | GOLD |
| `TASK_BRONZE_TO_SILVER` | Task (root) | FRAUD_ANALYSIS |
| `TASK_SILVER_TO_GOLD` | Task (child) | FRAUD_ANALYSIS |

In [ ]:
%%sql -r pipeline_tasks
SHOW TASKS IN SCHEMA KAGGLE_DB.FRAUD_ANALYSIS

In [ ]:
%%sql -r error_log
SELECT * FROM KAGGLE_DB.BRONZE.PIPELINE_ERROR_LOG ORDER BY CREATED_AT DESC LIMIT 10

---
## 9. Risk Scoring (Silver Layer Data Cleaning)

Three risk scores computed per transaction (0 = safe, 1 = risky):

| Score | Logic | Weight |
|-------|-------|--------|
| **Velocity Score** | # transactions by same customer in 24h window / 10 | 35% |
| **Location Score** | Distance to merchant / (3 × customer's avg distance) | 25% |
| **Amount Score** | \|Amount - customer avg\| / (3 × customer stddev) | 40% |

**Composite:** `RISK_SCORE = (AMOUNT × 0.40) + (VELOCITY × 0.35) + (LOCATION × 0.25)`

In [ ]:
%%sql -r risk_distribution
SELECT 
    CASE 
        WHEN RISK_SCORE < 0.2 THEN '0.0-0.2 (LOW)'
        WHEN RISK_SCORE < 0.4 THEN '0.2-0.4 (MODERATE)'
        WHEN RISK_SCORE < 0.6 THEN '0.4-0.6 (HIGH)'
        WHEN RISK_SCORE < 0.8 THEN '0.6-0.8 (VERY HIGH)'
        ELSE '0.8-1.0 (CRITICAL)'
    END AS RISK_BUCKET,
    COUNT(*) AS TOTAL,
    SUM(IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(AVG(VELOCITY_SCORE), 3) AS AVG_VELOCITY,
    ROUND(AVG(LOCATION_SCORE), 3) AS AVG_LOCATION,
    ROUND(AVG(AMOUNT_SCORE), 3) AS AVG_AMOUNT
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY RISK_BUCKET
ORDER BY RISK_BUCKET

In [ ]:
%%sql -r velocity_dist
SELECT 
    ROUND(VELOCITY_SCORE, 1) AS VELOCITY_BUCKET,
    COUNT(*) AS TOTAL,
    SUM(IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY VELOCITY_BUCKET
ORDER BY VELOCITY_BUCKET

In [ ]:
%%sql -r amount_dist
SELECT 
    ROUND(AMOUNT_SCORE, 1) AS AMOUNT_BUCKET,
    COUNT(*) AS TOTAL,
    SUM(IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY AMOUNT_BUCKET
ORDER BY AMOUNT_BUCKET

---
## 10. Outliers & Anomaly Detection

| Anomaly | Count | Fraud Rate | Insight |
|---------|-------|-----------|--------|
| Rapid-fire txns (≤5 min apart) | 11,424 | **1.37%** | Card-testing pattern |
| Amount outliers (IQR, >$193) | 27,778 | **5.81%** | Strongest single signal |
| High amount daytime (>$200) | 20,091 | **7.69%** | Highest fraud combo |
| Night + High amount | 5,258 | 1.26% | Lower than expected |
| Extreme distance (top 1%) | 5,558 | 0.43% | Distance alone is weak |
| Repeat fraud victims (10+ hits) | 97 customers | — | Targeted attacks |

In [ ]:
%%sql -r anomaly_patterns
SELECT 
    CASE WHEN LOCAL_HOUR BETWEEN 0 AND 5 AND AMT > 200 THEN 'NIGHT+HIGH_AMT'
         WHEN LOCAL_HOUR BETWEEN 0 AND 5 THEN 'NIGHT_ONLY'
         WHEN AMT > 200 THEN 'HIGH_AMT_ONLY'
         ELSE 'NORMAL'
    END AS PATTERN,
    COUNT(*) AS TOTAL,
    SUM(IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY PATTERN
ORDER BY FRAUD_RATE_PCT DESC

In [ ]:
%%sql -r rapid_fire
SELECT 
    COUNT(*) AS RAPID_FIRE_TXNS,
    SUM(IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(AVG(AMT), 2) AS AVG_AMOUNT
FROM (
    SELECT *,
        DATEDIFF('MINUTE', LAG(TRANS_DATE_TRANS_TIME) OVER (PARTITION BY CC_NUM ORDER BY TRANS_DATE_TRANS_TIME), TRANS_DATE_TRANS_TIME) AS MINS_SINCE_LAST
    FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
)
WHERE MINS_SINCE_LAST <= 5 AND MINS_SINCE_LAST >= 0

In [ ]:
%%sql -r amount_outliers
WITH stats AS (
    SELECT 
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY AMT) AS Q1,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY AMT) AS Q3
    FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
)
SELECT 
    COUNT(*) AS TOTAL_OUTLIERS,
    SUM(IS_FRAUD) AS FRAUD_IN_OUTLIERS,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(MIN(AMT), 2) AS MIN_AMT,
    ROUND(MAX(AMT), 2) AS MAX_AMT,
    ROUND(AVG(AMT), 2) AS AVG_AMT
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS, stats
WHERE AMT > Q3 + 1.5*(Q3-Q1)

In [ ]:
%%sql -r repeat_victims
SELECT 
    FRAUD_HITS,
    COUNT(*) AS CUSTOMER_COUNT,
    SUM(FRAUD_HITS) AS TOTAL_FRAUD_TXNS
FROM (
    SELECT CC_NUM, SUM(IS_FRAUD) AS FRAUD_HITS
    FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
    GROUP BY CC_NUM
    HAVING SUM(IS_FRAUD) > 0
) 
GROUP BY FRAUD_HITS
ORDER BY FRAUD_HITS DESC

In [ ]:
%%sql -r risky_merchants
SELECT 
    MERCHANT, CATEGORY,
    COUNT(*) AS TOTAL_TXNS,
    SUM(IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY MERCHANT, CATEGORY
HAVING SUM(IS_FRAUD) >= 5
ORDER BY FRAUD_RATE_PCT DESC
LIMIT 10

### Revised Key Findings

1. **High amount is the #1 fraud signal** — 7.69% fraud rate for >$200 (not nighttime)
2. **Nighttime alone is NOT risky** (0.02%) — the "late night = fraud" assumption is wrong
3. **97 customers are repeat targets** — one customer defrauded 19 times
4. **Rapid-fire card testing** (≤5 min gaps) shows clear pattern at 1.37%
5. **Online merchants** (`shopping_net`, `misc_net`) dominate top fraud rates
6. **Distance alone is weak** — top 1% distance only 0.43% fraud rate

### Recommendations Update
1. **Flag >$200 transactions** regardless of time — this is the strongest predictor
2. **Monitor rapid-fire patterns** — alert on 3+ txns within 5 minutes per card
3. **Protect repeat victims** — block cards after 3+ fraud hits, proactive outreach
4. **Increase scrutiny on online merchants** with >1.5% fraud rates
5. **Remove nighttime-only rules** — they create false positives without catching fraud

---
## 11. Merchant Score & Real-Time Fraud Alerts

### Merchant Score
Each merchant gets a risk score (0-1) based on historical fraud rate:
```
MERCHANT_SCORE = MIN(merchant_fraud_rate / 0.02, 1.0)
```
Where 0.02 (2%) is the threshold — merchants at or above 2% fraud rate get score = 1.0

| Merchant Risk | # Merchants | Fraud Rate |
|---------------|-------------|------------|
| HIGH RISK (0.8-1.0) | 16 | 1.88% |
| RISKY (0.5-0.8) | 53 | 1.20% |
| MODERATE (0.2-0.5) | 147 | 0.67% |
| SAFE (0-0.2) | 477 | 0.15% |

### Real-Time Alerts (Snowflake ALERT — every 1 minute)

| Alert | Trigger | Severity |
|-------|---------|----------|
| `ALERT_RAPID_FIRE_FRAUD` | 2+ txns from same card within 5 min | MEDIUM/HIGH/CRITICAL |
| `ALERT_HIGH_RISK_MERCHANT` | Merchant score ≥ 0.8 AND amount > $200 | CRITICAL |

Alerts are logged to `KAGGLE_DB.FRAUD_ANALYSIS.FRAUD_ALERTS` with full context.

In [ ]:
%%sql -r merchant_risk_dist
SELECT 
    CASE 
        WHEN MERCHANT_SCORE < 0.2 THEN 'SAFE (0-0.2)'
        WHEN MERCHANT_SCORE < 0.5 THEN 'MODERATE (0.2-0.5)'
        WHEN MERCHANT_SCORE < 0.8 THEN 'RISKY (0.5-0.8)'
        ELSE 'HIGH RISK (0.8-1.0)'
    END AS MERCHANT_RISK,
    COUNT(DISTINCT MERCHANT) AS MERCHANTS,
    COUNT(*) AS TXNS,
    SUM(IS_FRAUD) AS FRAUDS,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY MERCHANT_RISK
ORDER BY FRAUD_RATE_PCT DESC

In [ ]:
%%sql -r recent_alerts
SELECT * FROM KAGGLE_DB.FRAUD_ANALYSIS.FRAUD_ALERTS
ORDER BY CREATED_AT DESC
LIMIT 20

In [ ]:
%%sql -r active_alerts
SHOW ALERTS IN SCHEMA KAGGLE_DB.FRAUD_ANALYSIS

---
## 12. Cost Optimization & Credit Monitoring

### Baseline (Before Optimization) — May 8, 2026

| Metric | Value |
|--------|-------|
| Total Credits (7 days) | 6.75 |
| Warehouse Credits | 0.68 |
| COMPUTE_WH Auto-Suspend | 300 sec (default) |
| Alert Frequency | Every 1 min (2,880 checks/day) |
| Task Warehouse | COMPUTE_WH (shared) |
| Clustering | None |

### Optimizations Applied

| # | Change | Expected Impact |
|---|--------|----------------|
| 1 | COMPUTE_WH auto-suspend → 60 sec | ~80% idle credit reduction |
| 2 | Created FRAUD_TASK_WH (X-Small, auto-suspend 60s) | Isolated lightweight compute for tasks/alerts |
| 3 | Moved all tasks + alerts to FRAUD_TASK_WH | No longer wakes COMPUTE_WH for background jobs |
| 4 | Alert frequency → 5 min (from 1 min) | 5x fewer condition checks (576 vs 2,880/day) |
| 5 | Clustered FACT_TRANSACTIONS by (DATE_SK, CUSTOMER_SK) | Better partition pruning for time + customer queries |

In [ ]:
%%sql -r cost_breakdown_current
SELECT 
    service_type, 
    ROUND(SUM(credits_used), 2) AS total_credits, 
    ROUND(SUM(credits_used) / SUM(SUM(credits_used)) OVER () * 100, 1) AS pct_of_total 
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY 
WHERE start_time >= DATEADD(DAY, -7, CURRENT_DATE()) 
    AND start_time < CURRENT_DATE() 
GROUP BY service_type 
ORDER BY total_credits DESC

In [ ]:
%%sql -r warehouse_credits
SELECT 
    warehouse_name, 
    ROUND(SUM(credits_used), 4) AS credits,
    COUNT(*) AS metering_records
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY 
WHERE start_time >= DATEADD(DAY, -7, CURRENT_DATE()) 
    AND start_time < CURRENT_DATE() 
GROUP BY warehouse_name 
ORDER BY credits DESC

In [ ]:
%%sql -r daily_trend
SELECT 
    DATE_TRUNC('DAY', start_time) AS DAY,
    warehouse_name,
    ROUND(SUM(credits_used), 4) AS daily_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY 
WHERE start_time >= DATEADD(DAY, -14, CURRENT_DATE()) 
    AND start_time < CURRENT_DATE() 
GROUP BY DAY, warehouse_name
ORDER BY DAY DESC, daily_credits DESC

### How to Compare (Run After 7 Days)

Re-run the queries above after 7 days to compare:
- **Before:** 0.68 warehouse credits / 7 days (all on COMPUTE_WH)
- **After:** Expected < 0.20 warehouse credits / 7 days (split across COMPUTE_WH + FRAUD_TASK_WH)

**Key metrics to track:**
1. Total warehouse credits (should drop ~60-80%)
2. FRAUD_TASK_WH usage (should be minimal — XS + 60s suspend)
3. Alert execution count (should be ~5x lower)
4. Query performance on FACT_TRANSACTIONS (clustering should improve scan efficiency)

**Query to check clustering efficiency later:**
```sql
SELECT SYSTEM$CLUSTERING_INFORMATION('KAGGLE_DB.GOLD.FACT_TRANSACTIONS');
```

---
## 13. Architecture Notes & Maintenance Cost Projection

### Complete Architecture Summary

```
 SOURCE DATA
      |
      v
[BRONZE] RAW_TRANSACTIONS (Hybrid: 24 structured cols + RAW_DATA VARIANT)
      |                         ^
      | Stream: STM_RAW_TRANSACTIONS    | ALERT_SCHEMA_DRIFT (hourly)
      |                         |      | SP_SCHEMA_CHECKPOINT()
      v                         v      v
[SILVER] ENRICHED_TRANSACTIONS (35 cols: original + timezone + scores)
      |
      | Stream: STM_ENRICHED_TRANSACTIONS
      |
      v
[GOLD] Star Schema (1 Fact + 9 Dimensions)
      |
      v
 POWER BI / ANALYTICS
```

---

### Medallion Layers

| Layer | Table | Columns | Purpose |
|-------|-------|---------|--------|
| Bronze | `RAW_TRANSACTIONS` | 25 (24 + VARIANT) | Raw storage, zero transformation, schema-safe |
| Silver | `ENRICHED_TRANSACTIONS` | 35 | Cleansed, timezone-adjusted, risk-scored |
| Gold | `FACT_TRANSACTIONS` | 20 | Pure measures + 9 dimension FKs |
| Gold | `DIM_CUSTOMER` | 8 | 925 customers |
| Gold | `DIM_MERCHANT` | 6 | 700 merchants + merchant_score |
| Gold | `DIM_LOCATION` | 7 | 912 locations |
| Gold | `DIM_DATE` | 10 | 194 dates |
| Gold | `DIM_TIME_SLOT` | 5 | 24 hours |
| Gold | `DIM_AMOUNT_BUCKET` | 5 | 5 size ranges |
| Gold | `DIM_AGE_BUCKET` | 5 | 7 age groups |
| Gold | `DIM_RISK_BAND` | 6 | 5 risk levels |
| Gold | `DIM_TIMEZONE` | 5 | 6 US timezones |

---

### Pipeline Objects

| Object | Type | Schema | Schedule |
|--------|------|--------|----------|
| `STM_RAW_TRANSACTIONS` | Stream | BRONZE | CDC on Bronze |
| `STM_ENRICHED_TRANSACTIONS` | Stream | SILVER | CDC on Silver |
| `SP_BRONZE_TO_SILVER` | Procedure | SILVER | Called by task |
| `SP_SILVER_TO_GOLD` | Procedure | GOLD | Called by task |
| `SP_SCHEMA_CHECKPOINT` | Procedure | BRONZE | Called by alert |
| `SP_DETECT_RAPID_FIRE_FRAUD` | Procedure | FRAUD_ANALYSIS | Called by alert |
| `TASK_BRONZE_TO_SILVER` | Task (root) | FRAUD_ANALYSIS | Every 5 min |
| `TASK_SILVER_TO_GOLD` | Task (child) | FRAUD_ANALYSIS | After parent |
| `ALERT_RAPID_FIRE_FRAUD` | Alert | FRAUD_ANALYSIS | Every 5 min |
| `ALERT_HIGH_RISK_MERCHANT` | Alert | FRAUD_ANALYSIS | Every 5 min |
| `ALERT_SCHEMA_DRIFT` | Alert | FRAUD_ANALYSIS | Every 60 min |
| `PIPELINE_ERROR_LOG` | Table | BRONZE | Error tracking |
| `FRAUD_ALERTS` | Table | FRAUD_ANALYSIS | Alert log |
| `SCHEMA_REGISTRY` | Table | BRONZE | Schema drift log |

---

### Data Enrichments (Silver Layer)

| Feature | Logic |
|---------|-------|
| `TIMEZONE` | State → US timezone mapping |
| `LOCAL_TRANS_TIME` | UTC → local conversion |
| `LOCAL_HOUR` | Hour in customer's timezone |
| `AGE` | DOB → transaction date |
| `DISTANCE_KM` | HAVERSINE(customer, merchant) |
| `AMOUNT_BUCKET` | MICRO/SMALL/MEDIUM/LARGE/VERY_LARGE |
| `TIME_OF_DAY` | LATE_NIGHT/MORNING/AFTERNOON/EVENING |
| `VELOCITY_SCORE` | Txns in 24h window / 10 (0-1) |
| `LOCATION_SCORE` | Distance / (3 × avg distance) (0-1) |
| `AMOUNT_SCORE` | Z-score deviation / 3 (0-1) |
| `MERCHANT_SCORE` | Merchant fraud rate / 0.02 (0-1) |
| `RISK_SCORE` | Amount(40%) + Velocity(35%) + Location(25%) |

---

### Maintenance Cost Projection (1M records/day)

#### Storage

| Layer | Bytes/Row | Daily | Monthly | Monthly Cost |
|-------|-----------|-------|---------|-------------|
| Bronze | 104 | 99 MB | 2.9 GB | $0.07 |
| Silver | 95 | 91 MB | 2.7 GB | $0.06 |
| Gold | 66 | 63 MB | 1.9 GB | $0.04 |
| **Total** | **265** | **253 MB** | **7.5 GB** | **$0.17/month** |

#### Compute

| Operation | Warehouse | Frequency | Credits/Day |
|-----------|-----------|-----------|------------|
| TASK_BRONZE_TO_SILVER | FRAUD_TASK_WH (XS) | 288/day | ~0.08 |
| TASK_SILVER_TO_GOLD | FRAUD_TASK_WH (XS) | 288/day | ~0.12 |
| Alerts (3 total) | FRAUD_TASK_WH (XS) | 312/day | ~0.08 |
| Auto-clustering | Serverless | Continuous | ~0.05 |
| **Total** | | | **~0.33/day** |

#### Monthly Total

| Category | Cost |
|----------|------|
| Storage | $0.17 |
| Compute (~10 credits × $3) | $30 |
| Clustering | $5 |
| **Total** | **~$35/month** |

#### Annual Projection (365M rows)

| Metric | Value |
|--------|-------|
| Total storage | ~2.7 TB |
| Storage cost | ~$62/month |
| Compute | ~$35/month |
| **Annual total** | **~$1,164/year** |

---

### Schema Evolution (Hybrid VARIANT Approach)

**Current design:** Bronze has 24 structured columns + 1 `RAW_DATA VARIANT` column

| Scenario | What Happens |
|----------|-------------|
| Source adds new field | Stored in RAW_DATA automatically, no ALTER needed |
| Source removes field | Structured col stays (NULL for new rows), no break |
| Need new field in Silver | Extract from `RAW_DATA:field_name::TYPE` |
| Source schema changes | ALERT_SCHEMA_DRIFT detects within 60 min |

**Zero data loss guaranteed** — Bronze VARIANT captures everything.

---

### Error Handling

| Component | Error Handling |
|-----------|---------------|
| SP_BRONZE_TO_SILVER | TRY/CATCH → logs to PIPELINE_ERROR_LOG → RAISE |
| SP_SILVER_TO_GOLD | TRY/CATCH → logs to PIPELINE_ERROR_LOG → RAISE |
| SP_DETECT_RAPID_FIRE_FRAUD | TRY/CATCH → logs to PIPELINE_ERROR_LOG → RAISE |
| Tasks | Auto-suspend after repeated failures |
| Schema drift | Logged to SCHEMA_REGISTRY table |

---

### Optimization Applied

| # | Optimization | Impact |
|---|-------------|--------|
| 1 | COMPUTE_WH auto-suspend = 60s | ~80% idle reduction |
| 2 | Dedicated FRAUD_TASK_WH (XS) | Isolated lightweight compute |
| 3 | Alert frequency = 5 min | 5x fewer checks vs 1 min |
| 4 | Clustered FACT by (DATE_SK, CUSTOMER_SK) | Better partition pruning |
| 5 | Hybrid VARIANT in Bronze | Zero schema drift risk |
| 6 | DIM_MERCHANT fixed (700 rows vs 555K) | Proper star schema |

---

### Warehouses

| Warehouse | Size | Auto-Suspend | Purpose |
|-----------|------|--------------|--------|
| COMPUTE_WH | S | 60 sec | Ad-hoc queries, Power BI |
| FRAUD_TASK_WH | XS | 60 sec | Tasks, Alerts, Pipeline |

---

### Privileges Required

```sql
GRANT EXECUTE TASK ON ACCOUNT TO ROLE SYSADMIN;
GRANT EXECUTE MANAGED TASK ON ACCOUNT TO ROLE SYSADMIN;
```

---
## 14. Cost Analysis — Setup, Reduction & Strategy

### A. One-Time Setup Cost (What We Spent Building This)

| Phase | Operations | Est. Credits | Est. Cost ($3/credit) |
|-------|-----------|-------------|----------------------|
| Bronze layer (CTAS 555K rows) | 1 CTAS | 0.02 | $0.06 |
| Silver layer (CTAS + enrichments) | 1 CTAS + 7 UPDATEs | 0.15 | $0.45 |
| Gold layer (5 dims + 1 fact, rebuilt 3x) | ~10 CTAS/rebuilds | 0.10 | $0.30 |
| Risk scoring (4 UPDATE passes) | 4 UPDATEs on 555K rows | 0.08 | $0.24 |
| Timezone conversion (2 UPDATEs) | 2 UPDATEs on 555K rows | 0.04 | $0.12 |
| Merchant score (1 UPDATE) | 1 UPDATE | 0.02 | $0.06 |
| VARIANT backfill (1 UPDATE) | 1 UPDATE on 555K rows | 0.03 | $0.09 |
| Stored procedures (5 CREATEs) | DDL only | 0.00 | $0.00 |
| Tasks, Alerts, Streams (DDL) | DDL only | 0.00 | $0.00 |
| Testing & validation queries | ~50 SELECTs | 0.05 | $0.15 |
| **Total Setup** | | **~0.49 credits** | **~$1.47** |

---

### B. Before vs After Optimization

| Metric | Before (Baseline) | After (Optimized) | Savings |
|--------|-------------------|-------------------|--------|
| COMPUTE_WH auto-suspend | 300 sec | 60 sec | 80% idle reduction |
| Task warehouse | COMPUTE_WH (Small) | FRAUD_TASK_WH (XS) | 50% cheaper per second |
| Alert checks/day | 2,880 (1 min × 2) | 576 (5 min × 2) | **80% fewer executions** |
| Schema drift checks/day | N/A | 24 (hourly) | Minimal cost, max protection |
| Warehouse wake-ups/day | ~576 (shared WH) | ~300 (dedicated XS) | **48% fewer wake-ups** |
| Clustering | None | DATE_SK + CUSTOMER_SK | Faster scans = less compute |

---

### C. Daily Operational Cost Breakdown

#### Current State (555K rows, low activity)

| Component | Credits/Day | $/Day |
|-----------|-------------|-------|
| Tasks (2, every 5 min, skip if no data) | ~0.01 | $0.03 |
| Alerts (3, every 5-60 min) | ~0.01 | $0.03 |
| Ad-hoc queries (Power BI refresh) | ~0.05 | $0.15 |
| **Total** | **~0.07** | **$0.21/day** |

#### At Scale (1M records/day)

| Component | Credits/Day | $/Day |
|-----------|-------------|-------|
| TASK_BRONZE_TO_SILVER | 0.08 | $0.24 |
| TASK_SILVER_TO_GOLD | 0.12 | $0.36 |
| Alerts (condition checks) | 0.08 | $0.24 |
| Auto-clustering | 0.05 | $0.15 |
| Power BI refreshes (4/day) | 0.10 | $0.30 |
| **Total** | **0.43** | **$1.29/day** |

---

### D. Cost Reduction Strategies

#### Tier 1 — Quick Wins (No architecture changes)

| Strategy | How | Savings |
|----------|-----|--------|
| **Batch tasks to 30 min** | Change SCHEDULE from 5 min → 30 min | 83% fewer task runs |
| **Suspend alerts at night** | `ALTER ALERT ... SUSPEND` during 12AM-6AM | 25% fewer checks |
| **Use SYSTEM$STREAM_HAS_DATA** | Already done — tasks skip when no data | Avoids empty runs |
| **Resource monitor** | Set credit quota to prevent runaway | Cost cap protection |

#### Tier 2 — Architecture Changes

| Strategy | How | Savings |
|----------|-----|--------|
| **Transient Gold tables** | `CREATE TRANSIENT TABLE` — no fail-safe | 50% Gold storage savings |
| **Serverless tasks** | Remove warehouse, use serverless compute | Pay only for compute used (no 60s minimum) |
| **Dynamic tables** (replace tasks) | `CREATE DYNAMIC TABLE ... LAG = '5 MIN'` | Auto-managed, no SP overhead |
| **Materialized views** (Gold) | Auto-refresh with micro-partitions | Only recomputes changed data |

#### Tier 3 — At Scale (>10M/day)

| Strategy | How | Savings |
|----------|-----|--------|
| **Multi-cluster warehouse** | Auto-scale during peak loads | Avoids over-provisioning |
| **Query result caching** | Repeated Power BI queries use cache | ~30% compute reduction |
| **Search optimization** | For point lookups (CC_NUM, TRANS_NUM) | Faster lookups = less compute |
| **Data retention policy** | Archive Bronze >90 days to external stage | 70% storage savings on old data |
| **Aggregate tables** | Pre-computed daily/weekly summaries | Faster BI = fewer credits |

---

### E. Cost Comparison: Architecture Choices

| Design Choice | Monthly Cost (1M/day) | Why |
|---------------|----------------------|-----|
| **Current (Tasks + XS WH)** | ~$35 | Good balance of cost vs freshness |
| Dynamic Tables (replace tasks) | ~$25 | Snowflake optimizes refresh internally |
| Serverless Tasks | ~$20 | No idle time, pay per compute-second |
| Larger WH + less frequent | ~$45 | Faster but wasteful for small batches |
| Real-time (Snowpipe + 1 min tasks) | ~$80 | Lowest latency but highest cost |

---

### F. Recommended Cost Path

```
 NOW (Dev/Test)           PRODUCTION              AT SCALE (>10M/day)
 ─────────────           ──────────              ──────────────────
 $0.21/day               $1.29/day               $4-8/day
 
 • XS warehouse          • XS warehouse          • Small warehouse
 • 5 min tasks           • 15-30 min tasks       • Dynamic tables
 • 3 alerts              • Serverless tasks       • Multi-cluster
 • Manual refresh        • Scheduled BI refresh   • Result caching
                         • Resource monitor       • Data archival
                         • Night suspension       • Search optimization
```

---

### G. Monitoring Queries (Run Monthly)

```sql
-- Total credits this month
SELECT ROUND(SUM(credits_used), 2) 
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY 
WHERE start_time >= DATE_TRUNC('MONTH', CURRENT_DATE());

-- Credits by warehouse this month
SELECT warehouse_name, ROUND(SUM(credits_used), 2) AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY 
WHERE start_time >= DATE_TRUNC('MONTH', CURRENT_DATE())
GROUP BY warehouse_name ORDER BY credits DESC;

-- Task execution costs
SELECT task_name, COUNT(*) AS runs, ROUND(SUM(credits_used), 4) AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.SERVERLESS_TASK_HISTORY 
WHERE start_time >= DATE_TRUNC('MONTH', CURRENT_DATE())
GROUP BY task_name ORDER BY credits DESC;

-- Storage growth trend
SELECT usage_date, ROUND(SUM(average_database_bytes)/1024/1024/1024, 2) AS GB
FROM SNOWFLAKE.ACCOUNT_USAGE.DATABASE_STORAGE_USAGE_HISTORY
WHERE database_name = 'KAGGLE_DB'
GROUP BY usage_date ORDER BY usage_date DESC LIMIT 30;
```

---
## 15. Advanced Analysis

### A. Score Correlation — Which Feature Predicts Fraud Best?

| Score | Correlation with IS_FRAUD | Ranking |
|-------|--------------------------|--------|
| **AMOUNT_SCORE** | 0.1900 | #1 — Strongest predictor |
| **Raw AMT** | 0.1823 | #2 — Amount itself is powerful |
| **RISK_SCORE** (composite) | 0.0961 | #3 — Combined score |
| **MERCHANT_SCORE** | 0.0673 | #4 — Merchant history matters |
| LOCATION_SCORE | 0.0012 | #5 — Weak signal |
| VELOCITY_SCORE | 0.0006 | #6 — Very weak |
| DISTANCE_KM | 0.0002 | #7 — Nearly irrelevant |

**Insight:** Amount deviation is by far the strongest predictor. Location and velocity scores add minimal value individually — consider increasing AMOUNT_SCORE weight from 40% to 60%.

In [ ]:
%%sql -r correlation_matrix
SELECT 
    ROUND(CORR(VELOCITY_SCORE, IS_FRAUD), 4) AS VELOCITY_CORR,
    ROUND(CORR(LOCATION_SCORE, IS_FRAUD), 4) AS LOCATION_CORR,
    ROUND(CORR(AMOUNT_SCORE, IS_FRAUD), 4) AS AMOUNT_CORR,
    ROUND(CORR(MERCHANT_SCORE, IS_FRAUD), 4) AS MERCHANT_CORR,
    ROUND(CORR(RISK_SCORE, IS_FRAUD), 4) AS RISK_SCORE_CORR,
    ROUND(CORR(DISTANCE_KM, IS_FRAUD), 4) AS DISTANCE_CORR,
    ROUND(CORR(AMT, IS_FRAUD), 4) AS AMOUNT_RAW_CORR
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS

---
### B. Fraud by Day of Month — Salary Day Patterns

In [ ]:
%%sql -r fraud_by_day_of_month
SELECT 
    DAY(TRANS_DATE_TRANS_TIME) AS DAY_OF_MONTH,
    COUNT(*) AS TOTAL_TXNS,
    SUM(IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(SUM(CASE WHEN IS_FRAUD=1 THEN AMT ELSE 0 END), 2) AS FRAUD_LOSS
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY DAY_OF_MONTH
ORDER BY DAY_OF_MONTH

**Pattern:** Days 4-5 (0.69-0.70%) and 8 (0.62%) and 25 (0.60%) have elevated fraud rates.
Days 4-5 and 25 align with **salary deposit dates** — fraudsters know when accounts have money.

---
### C. Recency Analysis — Time Gap Before Fraud

In [ ]:
%%sql -r recency_analysis
SELECT 
    CASE 
        WHEN MINS_GAP < 5 THEN '< 5 min'
        WHEN MINS_GAP < 30 THEN '5-30 min'
        WHEN MINS_GAP < 60 THEN '30-60 min'
        WHEN MINS_GAP < 360 THEN '1-6 hours'
        WHEN MINS_GAP < 1440 THEN '6-24 hours'
        ELSE '> 24 hours'
    END AS TIME_SINCE_LAST_TXN,
    COUNT(*) AS FRAUD_TXNS,
    ROUND(AVG(AMT), 2) AS AVG_FRAUD_AMT
FROM (
    SELECT *,
        DATEDIFF('MINUTE', LAG(TRANS_DATE_TRANS_TIME) OVER (PARTITION BY CC_NUM ORDER BY TRANS_DATE_TRANS_TIME), TRANS_DATE_TRANS_TIME) AS MINS_GAP
    FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
    WHERE IS_FRAUD = 1
)
WHERE MINS_GAP IS NOT NULL
GROUP BY TIME_SINCE_LAST_TXN
ORDER BY MIN(MINS_GAP)

**Pattern:** Fraud occurs most in the **5-30 min window** (485 txns) and **1-6 hours** (545 txns).
Fraudsters test cards quickly (<5 min, $615 avg) then come back 1-6 hours later with larger amounts.

---
### D. Customer Segmentation — Victim Profiles

In [ ]:
%%sql -r victim_segments
SELECT 
    CASE 
        WHEN FRAUD_COUNT >= 10 THEN 'HEAVY_TARGET (10+)'
        WHEN FRAUD_COUNT >= 5 THEN 'REPEAT_VICTIM (5-9)'
        WHEN FRAUD_COUNT >= 2 THEN 'MULTI_HIT (2-4)'
        ELSE 'SINGLE_HIT (1)'
    END AS VICTIM_SEGMENT,
    COUNT(*) AS CUSTOMERS,
    SUM(FRAUD_COUNT) AS TOTAL_FRAUDS,
    ROUND(AVG(AVG_AMT), 2) AS AVG_TXN_AMOUNT,
    ROUND(AVG(AVG_FRAUD_AMT), 2) AS AVG_FRAUD_AMOUNT
FROM (
    SELECT CC_NUM, 
        SUM(IS_FRAUD) AS FRAUD_COUNT,
        AVG(AMT) AS AVG_AMT,
        AVG(CASE WHEN IS_FRAUD=1 THEN AMT END) AS AVG_FRAUD_AMT
    FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
    GROUP BY CC_NUM
    HAVING SUM(IS_FRAUD) > 0
)
GROUP BY VICTIM_SEGMENT
ORDER BY MIN(FRAUD_COUNT) DESC

**Key Finding:** 118 customers account for **1,440 frauds** (67% of all fraud) — these are systematically targeted.
Avg fraud amount is ~$525-$551 regardless of how often they're hit.

---
### E. Fraud Dollar Impact by Category

In [ ]:
%%sql -r fraud_dollar_impact
SELECT 
    CATEGORY,
    ROUND(SUM(CASE WHEN IS_FRAUD=1 THEN AMT ELSE 0 END), 2) AS TOTAL_FRAUD_LOSS,
    COUNT(CASE WHEN IS_FRAUD=1 THEN 1 END) AS FRAUD_TXNS,
    ROUND(AVG(CASE WHEN IS_FRAUD=1 THEN AMT END), 2) AS AVG_FRAUD_AMT,
    ROUND(MAX(CASE WHEN IS_FRAUD=1 THEN AMT END), 2) AS MAX_SINGLE_FRAUD
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY CATEGORY
HAVING FRAUD_TXNS > 0
ORDER BY TOTAL_FRAUD_LOSS DESC

**Insight:** `shopping_net` alone accounts for **$503K** (44% of total fraud loss) with avg fraud of $994.
Top 3 categories (`shopping_net`, `misc_net`, `shopping_pos`) = **80% of all fraud losses**.

---
### F. False Positive Estimation — Alert Tuning

| Threshold | Flagged | True Fraud | False Positives | Fraud Caught | FP Rate | Precision |
|-----------|---------|-----------|----------------|-------------|---------|----------|
| ≥ 0.4 | 113,105 | 1,409 | 111,696 | 65.7% | 98.8% | 1.2% |
| ≥ 0.5 | 33,577 | 1,099 | 32,478 | 51.2% | 96.7% | 3.3% |
| **≥ 0.6** | **7,901** | **651** | **7,250** | **30.4%** | **91.8%** | **8.2%** |
| ≥ 0.7 | 2,862 | 229 | 2,633 | 10.7% | 92.0% | 8.0% |

**Recommended threshold: 0.6**
- Catches 30% of fraud (651 out of 2,145)
- Only flags 7,901 transactions (1.4% of total traffic)
- Precision of 8.2% (1 in 12 flagged is actual fraud)
- Manageable for human review team

**For automated blocking: 0.7** — only 2,862 flagged, but catches fewer frauds.

In [ ]:
%%sql -r action_buckets
SELECT 
    CASE WHEN RISK_SCORE >= 0.7 THEN 'BLOCK (>=0.7)'
         WHEN RISK_SCORE >= 0.6 THEN 'REVIEW (0.6-0.7)'
         WHEN RISK_SCORE >= 0.4 THEN 'MONITOR (0.4-0.6)'
         ELSE 'PASS (<0.4)'
    END AS ACTION,
    COUNT(*) AS TRANSACTIONS,
    SUM(IS_FRAUD) AS FRAUD_CAUGHT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS PRECISION_PCT,
    ROUND(SUM(CASE WHEN IS_FRAUD=1 THEN AMT ELSE 0 END), 2) AS FRAUD_LOSS_PREVENTED
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY ACTION
ORDER BY MIN(RISK_SCORE) DESC

---
## 16. Pipeline Monitoring Dashboard (Execution Logs)

Both stored procedures now log every execution to `KAGGLE_DB.BRONZE.PIPELINE_EXECUTION_LOG`:
- Start/End timestamps
- Duration (seconds)
- Rows processed
- Execution status (SUCCESS / SKIPPED / FAILED)
- Warehouse used
- Batch timestamp

Use these queries for a **separate Power BI monitoring dashboard**:

In [ ]:
%%sql -r pipeline_history
-- Pipeline execution history (latest runs)
SELECT * FROM KAGGLE_DB.BRONZE.PIPELINE_EXECUTION_LOG
ORDER BY CREATED_AT DESC
LIMIT 50

In [ ]:
%%sql -r hourly_throughput
-- Hourly throughput: when does maximum data arrive?
SELECT 
    HOUR(BATCH_TIMESTAMP) AS HOUR_OF_DAY,
    COUNT(*) AS EXECUTIONS,
    SUM(ROWS_PROCESSED) AS TOTAL_ROWS,
    ROUND(AVG(ROWS_PROCESSED), 0) AS AVG_ROWS_PER_BATCH,
    ROUND(AVG(DURATION_SECONDS), 1) AS AVG_DURATION_SEC,
    MAX(ROWS_PROCESSED) AS PEAK_BATCH_SIZE
FROM KAGGLE_DB.BRONZE.PIPELINE_EXECUTION_LOG
WHERE EXECUTION_STATUS = 'SUCCESS'
GROUP BY HOUR_OF_DAY
ORDER BY HOUR_OF_DAY

In [ ]:
%%sql -r daily_health
-- Daily pipeline health summary
SELECT 
    DATE(BATCH_TIMESTAMP) AS RUN_DATE,
    PROCEDURE_NAME,
    COUNT(CASE WHEN EXECUTION_STATUS = 'SUCCESS' THEN 1 END) AS SUCCESS_RUNS,
    COUNT(CASE WHEN EXECUTION_STATUS = 'SKIPPED' THEN 1 END) AS SKIPPED_RUNS,
    COUNT(CASE WHEN EXECUTION_STATUS = 'FAILED' THEN 1 END) AS FAILED_RUNS,
    SUM(ROWS_PROCESSED) AS TOTAL_ROWS_PROCESSED,
    ROUND(AVG(DURATION_SECONDS), 1) AS AVG_DURATION_SEC,
    MAX(DURATION_SECONDS) AS MAX_DURATION_SEC
FROM KAGGLE_DB.BRONZE.PIPELINE_EXECUTION_LOG
GROUP BY RUN_DATE, PROCEDURE_NAME
ORDER BY RUN_DATE DESC, PROCEDURE_NAME

In [ ]:
%%sql -r lag_detection
-- Pipeline lag detection: time between batches
SELECT 
    PROCEDURE_NAME,
    BATCH_TIMESTAMP,
    ROWS_PROCESSED,
    DURATION_SECONDS,
    DATEDIFF('MINUTE', LAG(BATCH_TIMESTAMP) OVER (PARTITION BY PROCEDURE_NAME ORDER BY BATCH_TIMESTAMP), BATCH_TIMESTAMP) AS MINUTES_SINCE_LAST_RUN
FROM KAGGLE_DB.BRONZE.PIPELINE_EXECUTION_LOG
WHERE EXECUTION_STATUS = 'SUCCESS'
ORDER BY BATCH_TIMESTAMP DESC
LIMIT 20

In [ ]:
%%sql -r error_history
-- Error log (recent failures)
SELECT * FROM KAGGLE_DB.BRONZE.PIPELINE_ERROR_LOG
ORDER BY CREATED_AT DESC
LIMIT 10

### Monitoring Dashboard Tables for Power BI

| Table | Use In Dashboard |
|-------|------------------|
| `PIPELINE_EXECUTION_LOG` | Throughput trends, peak hours, SLA tracking |
| `PIPELINE_ERROR_LOG` | Error rates, failure patterns |
| `FRAUD_ALERTS` | Alert volume, severity distribution |
| `SCHEMA_REGISTRY` | Schema drift events |

### Key Metrics for Operations Dashboard

| Metric | Query From | Visualization |
|--------|-----------|---------------|
| Avg batch duration | PIPELINE_EXECUTION_LOG | Line chart (trend) |
| Peak data hour | PIPELINE_EXECUTION_LOG | Bar chart (hourly) |
| Failed runs/day | PIPELINE_EXECUTION_LOG | KPI card |
| Pipeline lag (min) | PIPELINE_EXECUTION_LOG | Gauge |
| Alerts/day by severity | FRAUD_ALERTS | Stacked bar |
| Schema drift events | SCHEMA_REGISTRY | Timeline |
| Rows processed/hour | PIPELINE_EXECUTION_LOG | Area chart |

### Connection for Ops Dashboard
- **Server:** `id84194.snowflakecomputing.com`
- **Warehouse:** `COMPUTE_WH`
- **Database:** `KAGGLE_DB`
- **Tables:** `BRONZE.PIPELINE_EXECUTION_LOG`, `BRONZE.PIPELINE_ERROR_LOG`, `FRAUD_ANALYSIS.FRAUD_ALERTS`, `BRONZE.SCHEMA_REGISTRY`

---
## 17. Additional Analysis

### A. Card Network Analysis

In [ ]:
%%sql -r card_network_fraud
SELECT 
    CASE 
        WHEN LEFT(CC_NUM, 1) = '4' THEN 'VISA'
        WHEN LEFT(CC_NUM, 1) = '5' THEN 'MASTERCARD'
        WHEN LEFT(CC_NUM, 1) = '3' THEN 'AMEX/DINERS'
        WHEN LEFT(CC_NUM, 1) = '6' THEN 'DISCOVER'
        WHEN LEFT(CC_NUM, 1) = '2' THEN 'MASTERCARD_NEW'
        ELSE 'OTHER'
    END AS CARD_NETWORK,
    COUNT(*) AS TOTAL_TXNS,
    SUM(IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(SUM(CASE WHEN IS_FRAUD=1 THEN AMT ELSE 0 END), 2) AS FRAUD_LOSS,
    COUNT(DISTINCT CC_NUM) AS UNIQUE_CARDS
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY CARD_NETWORK
ORDER BY FRAUD_LOSS DESC

**Insight:** AMEX/Diners cards account for **$470K loss** (41% of total) with highest fraud rate (0.43%). All networks have similar rates (0.34-0.44%) but AMEX has the most volume.

---
### B. Fraud Velocity — Weekly Trend (Is Fraud Increasing?)

In [ ]:
%%sql -r weekly_fraud_trend
SELECT 
    DATE_TRUNC('WEEK', TRANS_DATE_TRANS_TIME) AS WEEK_START,
    COUNT(*) AS TOTAL_TXNS,
    SUM(IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 3) AS FRAUD_RATE_PCT,
    ROUND(SUM(CASE WHEN IS_FRAUD=1 THEN AMT ELSE 0 END), 2) AS WEEKLY_FRAUD_LOSS
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY WEEK_START
ORDER BY WEEK_START

**Insight:** Fraud peaked in **Aug 3 week (0.74%)** and **Oct 12 week (0.73%)**, then dropped sharply in Dec.
Christmas week = 0.05% and New Year week = 0% — fraudsters take holidays too.

---
### C. Seasonal & Holiday Fraud Patterns

In [ ]:
%%sql -r seasonal_fraud
SELECT 
    CASE 
        WHEN DATE(TRANS_DATE_TRANS_TIME) BETWEEN '2020-11-25' AND '2020-11-30' THEN 'BLACK_FRIDAY_WEEK'
        WHEN DATE(TRANS_DATE_TRANS_TIME) BETWEEN '2020-12-20' AND '2020-12-25' THEN 'CHRISTMAS_WEEK'
        WHEN DATE(TRANS_DATE_TRANS_TIME) BETWEEN '2020-12-26' AND '2020-12-31' THEN 'NEW_YEAR_WEEK'
        WHEN DATE(TRANS_DATE_TRANS_TIME) BETWEEN '2020-07-01' AND '2020-07-05' THEN 'JULY_4TH'
        WHEN DATE(TRANS_DATE_TRANS_TIME) BETWEEN '2020-09-05' AND '2020-09-08' THEN 'LABOR_DAY'
        WHEN DATE(TRANS_DATE_TRANS_TIME) BETWEEN '2020-10-30' AND '2020-11-01' THEN 'HALLOWEEN'
        ELSE 'REGULAR'
    END AS PERIOD,
    COUNT(*) AS TOTAL_TXNS,
    SUM(IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(SUM(CASE WHEN IS_FRAUD=1 THEN AMT ELSE 0 END), 2) AS FRAUD_LOSS
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY PERIOD
ORDER BY FRAUD_RATE_PCT DESC

**Key findings:**
- **Labor Day** = highest fraud rate (0.61%) — people traveling, cards used in unusual locations
- **Black Friday** = 0.48% — high volume + elevated fraud
- **Christmas & New Year** = lowest (0.08% and 0%) — fraud drops dramatically in late Dec

---
### D. Fraud Hotspot Map (Geo-clustering)

In [ ]:
%%sql -r geo_hotspots
SELECT 
    STATE,
    ROUND(AVG(LAT), 4) AS CENTER_LAT,
    ROUND(AVG(LONG), 4) AS CENTER_LONG,
    COUNT(*) AS TOTAL_TXNS,
    SUM(IS_FRAUD) AS FRAUD_COUNT,
    ROUND(SUM(IS_FRAUD)*100.0/COUNT(*), 2) AS FRAUD_RATE_PCT,
    ROUND(SUM(CASE WHEN IS_FRAUD=1 THEN AMT ELSE 0 END), 2) AS FRAUD_LOSS
FROM KAGGLE_DB.SILVER.ENRICHED_TRANSACTIONS
GROUP BY STATE
ORDER BY FRAUD_LOSS DESC

**Top fraud hotspots:**
- **NY** = $99K loss (highest dollar impact)
- **AK** = 1.66% rate (highest rate but low volume)
- **CT** = 1.22% rate
- **ID** = 0.88% rate
- **5 states with ZERO fraud:** UT, WV, VT, NV, RI

Use `CENTER_LAT` and `CENTER_LONG` for Power BI map visual bubble sizing by `FRAUD_LOSS`.

---
## 18. Power BI Dashboard Plan

### Dashboard 1: Fraud Analytics (Main) — 5 Pages

| Page | Title | Visuals |
|------|-------|--------|
| 1 | **Executive Summary** | KPI cards (total fraud, rate, loss, avg amount), fraud trend line, top 5 categories bar |
| 2 | **Geographic Analysis** | US map (bubble by fraud loss), state table, timezone slicer |
| 3 | **Time & Pattern Analysis** | Fraud by local hour (bar), day-of-week (bar), day-of-month (line), seasonal heatmap |
| 4 | **Customer & Demographics** | Age group pie, gender split, victim segmentation, card network donut |
| 5 | **Risk Scoring & Alerts** | Risk band distribution, score correlation scatter, threshold tuning table, alert log |

### Dashboard 2: Operations Monitoring — 3 Pages

| Page | Title | Visuals |
|------|-------|--------|
| 6 | **Pipeline Health** | Execution timeline, success/fail/skip KPIs, duration trend, rows/batch |
| 7 | **Data Freshness & Lag** | Lag gauge, hourly throughput bar, peak hours heatmap |
| 8 | **Errors & Alerts** | Error table, alert volume by type, schema drift log |

### Total: 8 Pages (5 Analytics + 3 Operations)

---

### Slicers (Shared Across Pages)

| Slicer | Dimension Table | Field |
|--------|----------------|-------|
| Date Range | DIM_DATE | DATE_KEY |
| Time of Day | DIM_TIME_SLOT | TIME_OF_DAY |
| Amount Size | DIM_AMOUNT_BUCKET | AMOUNT_BUCKET |
| Age Group | DIM_AGE_BUCKET | AGE_GROUP |
| Risk Level | DIM_RISK_BAND | RISK_BAND |
| Timezone/Region | DIM_TIMEZONE | REGION |
| Category | DIM_MERCHANT | CATEGORY |
| State | DIM_LOCATION | STATE |

---

### DAX Measures Needed

```dax
// Core Measures
Fraud Count = CALCULATE(COUNTROWS(FACT_TRANSACTIONS), FACT_TRANSACTIONS[IS_FRAUD] = 1)
Total Transactions = COUNTROWS(FACT_TRANSACTIONS)
Fraud Rate % = DIVIDE([Fraud Count], [Total Transactions], 0) * 100
Total Fraud Loss = CALCULATE(SUM(FACT_TRANSACTIONS[AMT]), FACT_TRANSACTIONS[IS_FRAUD] = 1)
Avg Fraud Amount = DIVIDE([Total Fraud Loss], [Fraud Count], 0)

// Trend Measures
Fraud Count PW = CALCULATE([Fraud Count], DATEADD(DIM_DATE[DATE_KEY], -7, DAY))
Fraud Rate WoW % = DIVIDE([Fraud Count] - [Fraud Count PW], [Fraud Count PW], 0) * 100

// Risk Measures
Avg Risk Score = AVERAGE(FACT_TRANSACTIONS[RISK_SCORE])
High Risk Transactions = CALCULATE(COUNTROWS(FACT_TRANSACTIONS), FACT_TRANSACTIONS[RISK_SCORE] >= 0.6)
```

---

### Data Model in Power BI

```
         DIM_DATE ─────────┐
         DIM_TIME_SLOT ─────┤
         DIM_AGE_BUCKET ────┤
         DIM_AMOUNT_BUCKET ─┤
         DIM_RISK_BAND ─────┼── FACT_TRANSACTIONS (555K)
         DIM_TIMEZONE ──────┤
         DIM_CUSTOMER ──────┤
         DIM_MERCHANT ──────┤
         DIM_LOCATION ──────┘

    (Separate) PIPELINE_EXECUTION_LOG → Ops Dashboard
    (Separate) FRAUD_ALERTS → Alert Dashboard
```

### Connection Settings

| Setting | Value |
|---------|-------|
| Server | `id84194.snowflakecomputing.com` |
| Warehouse | `COMPUTE_WH` |
| Database | `KAGGLE_DB` |
| Schema | `GOLD` (analytics), `BRONZE` (ops monitoring) |
| Mode | Import (for dimensions) + DirectQuery (for fact) |
| Refresh | Scheduled 4x/day |

---
## 19. Power BI Connection Layer (Views)

### Strategy: Import Mode Only (Snowflake Trial)

Since we're on Snowflake Trial (no DirectQuery/Hybrid available), we use **Import mode** with pre-aggregated views to minimize refresh cost.

**How it works:**
- Power BI imports data from views on scheduled refresh
- Pre-aggregated views return 5-50 rows (not 555K) → fast & cheap
- Snowflake result cache means repeated queries = $0 for 24h
- Estimated cost: **~$0.03 per refresh** (~$3.60/month at 4 refreshes/day)

---

### Views Created (17 total in `KAGGLE_DB.GOLD`)

#### Dimension Views (Import these — tiny, rarely change)

| View | Rows | Refresh Cost |
|------|------|--------------|
| `VW_PBI_DIM_DATE` | 194 | ~$0 |
| `VW_PBI_DIM_CUSTOMER` | 925 | ~$0 |
| `VW_PBI_DIM_MERCHANT` | 700 | ~$0 |
| `VW_PBI_DIM_LOCATION` | 912 | ~$0 |
| `VW_PBI_DIM_TIME_SLOT` | 24 | ~$0 |
| `VW_PBI_DIM_AMOUNT_BUCKET` | 5 | ~$0 |
| `VW_PBI_DIM_AGE_BUCKET` | 7 | ~$0 |
| `VW_PBI_DIM_RISK_BAND` | 5 | ~$0 |
| `VW_PBI_DIM_TIMEZONE` | 6 | ~$0 |

#### Pre-Aggregated Views (Import these — small results, pre-computed)

| View | Returns | Use For |
|------|---------|--------|
| `VW_PBI_EXEC_SUMMARY` | 1 row | KPI cards (total fraud, rate, loss) |
| `VW_PBI_FRAUD_BY_CATEGORY` | 14 rows | Category bar chart |
| `VW_PBI_FRAUD_BY_HOUR` | 24 rows | Hourly line chart |
| `VW_PBI_FRAUD_BY_STATE` | 50 rows | Map visual |
| `VW_PBI_FRAUD_BY_DEMOGRAPHICS` | ~14 rows | Age/Gender matrix |
| `VW_PBI_MONTHLY_TREND` | 7 rows | Monthly trend line |
| `VW_PBI_RISK_DISTRIBUTION` | 5 rows | Risk band donut |

#### Fact View (Import — full dataset for drill-through)

| View | Rows | Use For |
|------|------|---------|
| `VW_PBI_FACT_TRANSACTIONS` | 555K | Custom slicing, drill-through, detail pages |

---

### Why Regular Views (Not Materialized)?

| Factor | Regular View | Materialized View |
|--------|-------------|-------------------|
| Storage cost | $0 | Extra storage + auto-refresh |
| Freshness | Always real-time | Lag until refresh |
| At 555K rows | Scans in <1 sec | Overkill |
| Pre-aggregated (5-50 rows) | Snowflake caches 24h | Not needed |
| When to upgrade | >100M rows | Future consideration |

---

### Power BI Setup Steps (Import Mode)

1. **Get Data** → Snowflake connector
2. **Server:** `id84194.snowflakecomputing.com`
3. **Warehouse:** `COMPUTE_WH`
4. **Database:** `KAGGLE_DB`
5. **Import these views:**
   - All `VW_PBI_DIM_*` (9 dimension views)
   - All `VW_PBI_FRAUD_*` + `VW_PBI_EXEC_SUMMARY` + `VW_PBI_RISK_*` + `VW_PBI_MONTHLY_*` (7 aggregated views)
   - `VW_PBI_FACT_TRANSACTIONS` (1 fact view)
6. **Create relationships** on `*_SK` columns between fact and dims
7. **Set refresh schedule:** 4x/day (every 6 hours)

---

### Refresh Cost Breakdown

| Refresh Component | Data Scanned | Time | Credits |
|-------------------|-------------|------|---------|
| 9 dimension views | ~3,000 rows total | <1 sec | ~0.001 |
| 7 aggregated views | ~120 rows returned | <1 sec | ~0.005 |
| 1 fact view | 555K rows | ~1-2 sec | ~0.02 |
| **Total per refresh** | | **~2-3 sec** | **~0.026** |

| Schedule | Refreshes/Month | Monthly Cost |
|----------|----------------|--------------|
| 1x/day | 30 | $2.34 |
| 4x/day | 120 | $9.36 |
| 8x/day | 240 | $18.72 |

---

### Cost Optimization Tips for Trial

1. **Use aggregated views first** — avoid importing full fact unless needed for drill-through
2. **Result cache** — if you refresh within 24h and data hasn't changed, cost = $0
3. **Incremental refresh** — configure in Power BI to only pull new rows (by date)
4. **Suspend COMPUTE_WH** after hours — no credits burned when not in use
5. **Schedule refresh during off-peak** — fewer concurrent queries = faster completion

---
## 20. Power BI Report Blueprint — Page-by-Page Design

---

### PAGE 0: Home / Navigation — "Welcome & Overview"

**Purpose:** Landing page with navigation buttons, last refresh timestamp, and quick health indicators.

| Visual | Type | Details |
|--------|------|--------|
| Report Title | Text Box | "Credit Card Fraud Intelligence Dashboard" |
| Last Data Refresh | Card | MAX(TRANS_DATE_TRANS_TIME) from fact |
| Navigation Buttons (7) | Buttons with page navigation | Each button labeled: When, Where, Who, Why, How, Alerts, Ops |
| Mini KPIs (top bar) | Card Row | Total Fraud: 2,145 | Rate: 0.39% | Loss: $1.13M | Active Alerts: [count] |
| Fraud Trend Sparkline | Sparkline/Small Line | Monthly fraud rate (7 data points) |
| Risk Gauge | Gauge | Current avg RISK_SCORE across all transactions |
| System Status Indicator | Icon + Text | Pipeline: ✓ Running | Last batch: [timestamp] |

**Design Notes:**
- Dark theme background (navy/charcoal)
- Color coding: Red = Critical, Orange = Warning, Green = Safe
- Each navigation button shows a mini-stat preview
- No slicers on home page — clean landing

**Layout:**
```
┌─────────────────────────────────────────────────────┐
│  CREDIT CARD FRAUD INTELLIGENCE DASHBOARD           │
│  Last Refresh: May 8, 2026 07:30 AM                 │
├─────────────────────────────────────────────────────┤
│                                                     │
│  [2,145]    [0.39%]    [$1.13M]    [218 Customers]  │
│  Frauds     Rate       Loss        Affected         │
│                                                     │
├────────┬────────┬────────┬────────┬────────┬────────┤
│        │        │        │        │        │        │
    │  WHEN  │ WHERE  │  WHO   │  WHY   │  HOW   │ ALERTS │
│  ⏰    │  🗺️   │  👤   │  ❓   │  🔍   │  🚨   │
│ Peak:  │ Top:   │ Age:   │ Score: │ Online │ Active │
│ 5-7PM  │  NY    │  55+   │ 0.19   │  44%   │  [n]   │
│        │        │        │        │        │        │
├────────┴────────┴────────┴────────┴────────┴────────┤
│                                                     │
│  [Monthly Fraud Trend Line]        [Risk Gauge]     │
│                                                     │
│  Pipeline: ✓ Healthy    │    Schema: ✓ No Drift    │
└─────────────────────────────────────────────────────┘
```

---

### PAGE 1: Executive Summary — "WHAT is happening?"

**Purpose:** High-level KPIs for leadership. Answer: How much fraud? What's the trend? How bad is it?

| Visual | Type | Source View | Metric |
|--------|------|-------------|--------|
| Total Transactions | KPI Card | VW_PBI_EXEC_SUMMARY | TOTAL_TRANSACTIONS |
| Total Fraud Cases | KPI Card | VW_PBI_EXEC_SUMMARY | TOTAL_FRAUDS |
| Fraud Rate % | KPI Card | VW_PBI_EXEC_SUMMARY | FRAUD_RATE_PCT |
| Total Fraud Loss ($) | KPI Card | VW_PBI_EXEC_SUMMARY | TOTAL_FRAUD_LOSS |
| Avg Fraud Amount | KPI Card | VW_PBI_EXEC_SUMMARY | AVG_FRAUD_AMOUNT |
| Affected Customers | KPI Card | VW_PBI_EXEC_SUMMARY | AFFECTED_CUSTOMERS |
| Monthly Trend | Line Chart | VW_PBI_MONTHLY_TREND | MONTH_NAME (X), FRAUD_COUNT (Y), FRAUD_RATE_PCT (secondary Y) |
| Fraud by Category (Top 5) | Horizontal Bar | VW_PBI_FRAUD_BY_CATEGORY | CATEGORY (Y), FRAUD_LOSS (X) |
| Risk Band Split | Donut Chart | VW_PBI_RISK_DISTRIBUTION | RISK_BAND (legend), TOTAL_TXNS (values) |

**Slicers:** Date Range, Timezone Region

---

### PAGE 2: When — "WHEN does fraud happen?"

**Purpose:** Temporal patterns. Answer: What hour? What day? What season? When should we be most alert?

| Visual | Type | Source View | Metric |
|--------|------|-------------|--------|
| Fraud by Local Hour | Column Chart | VW_PBI_FRAUD_BY_HOUR | HOUR_OF_DAY (X), FRAUD_RATE_PCT (Y), color by TIME_OF_DAY |
| Business Hours vs After Hours | Stacked Bar | VW_PBI_FRAUD_BY_HOUR | IS_BUSINESS_HOURS (X), FRAUD_COUNT (Y) |
| Day of Week | Bar Chart | VW_PBI_FACT_TRANSACTIONS + DIM_DATE | DAY_NAME (Y), FRAUD_COUNT (X) |
| Monthly Seasonality | Area Chart | VW_PBI_MONTHLY_TREND | MONTH_NAME (X), FRAUD_RATE_PCT (Y) |
| Holiday Impact Table | Table | Custom query (Section 17C) | PERIOD, FRAUD_RATE_PCT, FRAUD_LOSS |

**Key Insight Callout:** "Peak fraud: 5-7 PM local time (1.7%). Lowest: Christmas week (0.08%)"

**Slicers:** Timezone, Amount Bucket

---

### PAGE 3: Where — "WHERE does fraud happen?"

**Purpose:** Geographic patterns. Answer: Which states? Which regions? Urban vs rural?

| Visual | Type | Source View | Metric |
|--------|------|-------------|--------|
| US Map (Bubbles) | Filled Map / Bubble Map | VW_PBI_FRAUD_BY_STATE | STATE (location), FRAUD_LOSS (bubble size), FRAUD_RATE_PCT (color) |
| Top 10 States by Loss | Bar Chart | VW_PBI_FRAUD_BY_STATE | STATE (Y), FRAUD_LOSS (X) |
| Top 10 States by Rate | Bar Chart | VW_PBI_FRAUD_BY_STATE | STATE (Y), FRAUD_RATE_PCT (X) |
| Timezone Region Split | Pie Chart | VW_PBI_DIM_TIMEZONE joined with fact | REGION (legend), FRAUD_COUNT (values) |
| Zero-Fraud States | Card | VW_PBI_FRAUD_BY_STATE | States where FRAUD_COUNT = 0 |

**Key Insight Callout:** "NY = $99K loss (highest $). AK = 1.66% rate (highest %). 5 states = zero fraud."

**Slicers:** Risk Band, Category

---

### PAGE 4: Who — "WHO is being targeted?"

**Purpose:** Victim & demographic analysis. Answer: Which customers? What age? What gender? What card?

| Visual | Type | Source View | Metric |
|--------|------|-------------|--------|
| Age Group Distribution | Clustered Bar | VW_PBI_FRAUD_BY_DEMOGRAPHICS | AGE_GROUP (Y), FRAUD_RATE_PCT (X), color by GENDER |
| Gender Split | Donut | VW_PBI_FRAUD_BY_DEMOGRAPHICS | GENDER (legend), FRAUD_COUNT (values) |
| Generation Analysis | Stacked Column | VW_PBI_FRAUD_BY_DEMOGRAPHICS | GENERATION (X), FRAUD_COUNT (Y), color by GENDER |
| Card Network | Bar Chart | Custom query (Section 17A) | CARD_NETWORK (Y), FRAUD_LOSS (X) |
| Repeat Victims | Table | Custom query (Section 15D) | VICTIM_SEGMENT, CUSTOMERS, TOTAL_FRAUDS |
| Top Targeted Customers | Table | VW_PBI_FACT_TRANSACTIONS | CUSTOMER_SK, COUNT(IS_FRAUD), SUM(AMT) |

**Key Insight Callout:** "118 heavy targets = 67% of all fraud. AMEX = $470K loss (41% of total)."

**Slicers:** Age Group, Gender, Card Network

---

### PAGE 5: Why — "WHY did fraud succeed? (Risk Factors)"

**Purpose:** Root cause analysis. Answer: What patterns predict fraud? What's the risk profile?

| Visual | Type | Source View | Metric |
|--------|------|-------------|--------|
| Risk Score Distribution | Histogram / Column | VW_PBI_RISK_DISTRIBUTION | RISK_BAND (X), TOTAL_TXNS (Y), FRAUD_RATE_PCT (data label) |
| Amount vs Fraud Rate | Scatter | VW_PBI_FACT_TRANSACTIONS | AMT (X), RISK_SCORE (Y), IS_FRAUD (color) |
| Score Correlation Table | Table | Custom (Section 15A) | Score name, Correlation value |
| Fraud Pattern Combo | Matrix | Custom (Section 10) | PATTERN, FRAUD_RATE_PCT |
| Threshold Impact | Table | Custom (Section 15F) | THRESHOLD, FLAGGED, FRAUD_CAUGHT, PRECISION |
| Action Buckets | Funnel | Custom (Section 15F) | ACTION → BLOCK/REVIEW/MONITOR/PASS |

**Key Insight Callout:** "Amount deviation = #1 predictor (0.19 corr). Threshold 0.6 catches 30% fraud at 8% precision."

**Slicers:** Risk Band, Amount Bucket

---

### PAGE 6: How — "HOW do fraudsters operate? (Patterns & Methods)"

**Purpose:** Behavioral patterns. Answer: What are the attack methods? How do they test cards?

| Visual | Type | Source View | Metric |
|--------|------|-------------|--------|
| Category Fraud Loss (Treemap) | Treemap | VW_PBI_FRAUD_BY_CATEGORY | CATEGORY (group), FRAUD_LOSS (size), FRAUD_RATE_PCT (color) |
| Top Risky Merchants | Table | VW_PBI_DIM_MERCHANT | MERCHANT, CATEGORY, MERCHANT_SCORE (top 10) |
| Rapid-Fire Pattern | KPI + Sparkline | FRAUD_ALERTS table | Count of RAPID_FIRE alerts |
| Recency (Time Gap) | Bar Chart | Custom (Section 15C) | TIME_SINCE_LAST_TXN (Y), FRAUD_TXNS (X) |
| Online vs In-Person | Stacked Bar | VW_PBI_FRAUD_BY_CATEGORY | Online (_net) vs POS categories |
| Day-of-Month Pattern | Line Chart | Custom (Section 17B) | DAY_OF_MONTH (X), FRAUD_RATE_PCT (Y) |

**Key Insight Callout:** "shopping_net = $503K (44% of loss). Salary days (4-5, 25) = highest fraud. Card testing in <5 min."

**Slicers:** Category, Merchant Score Band

---

### PAGE 7: Alerts & Actions — "What SHOULD we do?"

**Purpose:** Operational response. Answer: What's the alert status? What needs action?

| Visual | Type | Source View | Metric |
|--------|------|-------------|--------|
| Active Alerts Count | KPI Cards | FRAUD_ALERTS | Count by ALERT_SEVERITY (CRITICAL, HIGH, MEDIUM) |
| Recent Alerts Table | Table | FRAUD_ALERTS | ALERT_TYPE, CC_NUM, MERCHANT, AMT, SEVERITY, CREATED_AT |
| Alert Volume by Day | Line Chart | FRAUD_ALERTS | DATE(CREATED_AT) (X), COUNT (Y), color by ALERT_TYPE |
| Unresolved Alerts | Gauge | FRAUD_ALERTS | WHERE IS_RESOLVED = FALSE |
| Recommended Actions | Table | VW_PBI_RISK_DISTRIBUTION | RISK_BAND, ACTION_REQUIRED, FRAUD_RATE_PCT |
| Dollar at Risk | KPI | VW_PBI_FACT_TRANSACTIONS | SUM(AMT) WHERE RISK_SCORE >= 0.6 AND IS_FRAUD = 0 |

**Slicers:** Alert Type, Severity, Date Range

---

### PAGE 8: Pipeline Operations — "Is the system HEALTHY?"

**Purpose:** Technical monitoring. Answer: Is data flowing? Any failures? What's the lag?

| Visual | Type | Source View | Metric |
|--------|------|-------------|--------|
| Pipeline Status | Traffic Light KPI | PIPELINE_EXECUTION_LOG | Latest status (SUCCESS = green) |
| Execution Timeline | Gantt / Line | PIPELINE_EXECUTION_LOG | START_TIME (X), DURATION_SECONDS (Y) |
| Success/Fail/Skip | Stacked Column | PIPELINE_EXECUTION_LOG | DATE (X), COUNT (Y), color by STATUS |
| Avg Batch Duration | KPI Card | PIPELINE_EXECUTION_LOG | AVG(DURATION_SECONDS) |
| Rows Processed Trend | Area Chart | PIPELINE_EXECUTION_LOG | BATCH_TIMESTAMP (X), ROWS_PROCESSED (Y) |
| Error Log | Table | PIPELINE_ERROR_LOG | PROCEDURE_NAME, ERROR_MESSAGE, CREATED_AT |
| Schema Drift Events | Table | SCHEMA_REGISTRY | COLUMN_NAME, CHANGE_TYPE, DETECTED_AT |

**Slicers:** Procedure Name, Date Range

---

### Summary: 9 Pages (Home + 8 Analysis)

| Page | Question | Audience |
|------|----------|----------|
| 0. Home | Navigation & Quick Stats | Everyone |
| 1. Executive Summary | **WHAT** is happening? | C-Suite, Management |
| 2. When | **WHEN** does fraud peak? | Fraud Analysts |
| 3. Where | **WHERE** are the hotspots? | Regional Teams |
| 4. Who | **WHO** is being targeted? | Customer Protection |
| 5. Why | **WHY** did fraud succeed? | Data Scientists |
| 6. How | **HOW** do fraudsters operate? | Investigators |
| 7. Alerts & Actions | **What to DO?** | Operations Team |
| 8. Pipeline Health | **Is system OK?** | Engineering/DevOps |

---
## 21. Power BI Theme & Build Guide

### Step 1: Create Theme JSON File

Save this as `Fraud_Dark_Theme.json` on your computer:

```json
{
  "name": "Fraud Intelligence Dark",
  "dataColors": [
    "#4299E1",
    "#38B2AC",
    "#F6AD55",
    "#FC8181",
    "#E53E3E",
    "#9F7AEA",
    "#48BB78",
    "#ED8936"
  ],
  "background": "#1B2838",
  "foreground": "#FFFFFF",
  "tableAccent": "#4299E1",
  "good": "#38B2AC",
  "neutral": "#F6AD55",
  "bad": "#E53E3E",
  "maximum": "#E53E3E",
  "center": "#F6AD55",
  "minimum": "#38B2AC",
  "visualStyles": {
    "*": {
      "*": {
        "background": [{"color": {"solid": {"color": "#2D3748"}}}],
        "outlineColor": [{"solid": {"color": "#4A5568"}}],
        "border": [{"color": {"solid": {"color": "#4A5568"}}, "show": true}]
      }
    },
    "page": {
      "*": {
        "background": [{"color": {"solid": {"color": "#1B2838"}}, "transparency": 0}]
      }
    }
  },
  "textClasses": {
    "title": {"color": "#FFFFFF", "fontSize": 14, "fontFace": "Segoe UI Semibold"},
    "header": {"color": "#FFFFFF", "fontSize": 12, "fontFace": "Segoe UI"},
    "label": {"color": "#A0AEC0", "fontSize": 10, "fontFace": "Segoe UI"},
    "callout": {"color": "#FFFFFF", "fontSize": 24, "fontFace": "Segoe UI Light"}
  }
}
```

---

### Step 2: Import Theme in Power BI Desktop

1. Open **Power BI Desktop**
2. Go to **View** tab (top ribbon)
3. Click **Themes** dropdown → **Browse for themes**
4. Select your `Fraud_Dark_Theme.json` file
5. Click **Open** → Theme applied to all pages

---

### Step 3: Connect to Snowflake

1. **Home** → **Get Data** → **Snowflake**
2. Enter:
   - Server: `id84194.snowflakecomputing.com`
   - Warehouse: `COMPUTE_WH`
3. Click **OK** → Sign in (Microsoft account or credentials)
4. **Navigator** → expand `KAGGLE_DB` → `GOLD`
5. Select all `VW_PBI_*` views (check all 17)
6. Click **Load** (Import mode)

---

### Step 4: Create Relationships (Model View)

1. Switch to **Model** view (left sidebar, 3rd icon)
2. Create these relationships (drag & drop):

| From (Fact) | To (Dimension) | Key |
|-------------|---------------|-----|
| VW_PBI_FACT_TRANSACTIONS.CUSTOMER_SK | VW_PBI_DIM_CUSTOMER.CUSTOMER_SK | Many-to-One |
| VW_PBI_FACT_TRANSACTIONS.MERCHANT_SK | VW_PBI_DIM_MERCHANT.MERCHANT_SK | Many-to-One |
| VW_PBI_FACT_TRANSACTIONS.LOCATION_SK | VW_PBI_DIM_LOCATION.LOCATION_SK | Many-to-One |
| VW_PBI_FACT_TRANSACTIONS.DATE_SK | VW_PBI_DIM_DATE.DATE_SK | Many-to-One |
| VW_PBI_FACT_TRANSACTIONS.TIME_SLOT_SK | VW_PBI_DIM_TIME_SLOT.TIME_SLOT_SK | Many-to-One |
| VW_PBI_FACT_TRANSACTIONS.AMOUNT_BUCKET_SK | VW_PBI_DIM_AMOUNT_BUCKET.AMOUNT_BUCKET_SK | Many-to-One |
| VW_PBI_FACT_TRANSACTIONS.AGE_BUCKET_SK | VW_PBI_DIM_AGE_BUCKET.AGE_BUCKET_SK | Many-to-One |
| VW_PBI_FACT_TRANSACTIONS.RISK_BAND_SK | VW_PBI_DIM_RISK_BAND.RISK_BAND_SK | Many-to-One |
| VW_PBI_FACT_TRANSACTIONS.TIMEZONE_SK | VW_PBI_DIM_TIMEZONE.TIMEZONE_SK | Many-to-One |

---

### Step 5: Create DAX Measures

In **Report** view → right-click `VW_PBI_FACT_TRANSACTIONS` → **New Measure**:

```dax
// Core Measures
Fraud Count = CALCULATE(COUNTROWS(VW_PBI_FACT_TRANSACTIONS), VW_PBI_FACT_TRANSACTIONS[IS_FRAUD] = 1)

Total Transactions = COUNTROWS(VW_PBI_FACT_TRANSACTIONS)

Fraud Rate % = DIVIDE([Fraud Count], [Total Transactions], 0) * 100

Total Fraud Loss = CALCULATE(SUM(VW_PBI_FACT_TRANSACTIONS[AMT]), VW_PBI_FACT_TRANSACTIONS[IS_FRAUD] = 1)

Avg Fraud Amount = DIVIDE([Total Fraud Loss], [Fraud Count], 0)

Affected Customers = CALCULATE(DISTINCTCOUNT(VW_PBI_FACT_TRANSACTIONS[CUSTOMER_SK]), VW_PBI_FACT_TRANSACTIONS[IS_FRAUD] = 1)

// Risk Measures
Avg Risk Score = AVERAGE(VW_PBI_FACT_TRANSACTIONS[RISK_SCORE])

High Risk Txns = CALCULATE(COUNTROWS(VW_PBI_FACT_TRANSACTIONS), VW_PBI_FACT_TRANSACTIONS[RISK_SCORE] >= 0.6)

// Legit vs Fraud Avg Amount
Avg Legit Amount = CALCULATE(AVERAGE(VW_PBI_FACT_TRANSACTIONS[AMT]), VW_PBI_FACT_TRANSACTIONS[IS_FRAUD] = 0)
```

---

### Step 6: Build Pages (Order)

#### Page 0 — Home
1. Set page background: `#1B2838` (Format pane → Page → Canvas background)
2. Add **Text box**: "Credit Card Fraud Intelligence Dashboard"
3. Add **6 Cards** in a row: Fraud Count, Rate, Loss, Avg Amount, Customers, High Risk Txns
4. Add **Buttons** (Insert → Buttons → Blank): one per page, set Action → Page Navigation
5. Add **Line chart** (small): VW_PBI_MONTHLY_TREND → MONTH_NAME (X), FRAUD_RATE_PCT (Y)

#### Page 1 — Executive Summary
1. Add **6 KPI Cards** (top row): use DAX measures
2. Add **Line chart**: VW_PBI_MONTHLY_TREND (trend)
3. Add **Bar chart**: VW_PBI_FRAUD_BY_CATEGORY (top categories)
4. Add **Donut**: VW_PBI_RISK_DISTRIBUTION (risk split)
5. Add **Slicers**: DIM_DATE[DATE_KEY], DIM_TIMEZONE[REGION]

#### Pages 2-8 — Follow Section 20 Blueprint
- Each page: set title text box, add visuals per the table, add slicers
- Use **conditional formatting** on tables: red for high fraud rate, green for low

---

### Step 7: Format Tips

| Setting | Where | Value |
|---------|-------|-------|
| Card background | Format → Background | `#2D3748` |
| Card border | Format → Border | `#4A5568`, rounded |
| Font color | Format → Data label | `#FFFFFF` |
| Title color | Format → Title | `#FFFFFF` |
| Axis labels | Format → X/Y axis | `#A0AEC0` |
| Gridlines | Format → Gridlines | OFF (cleaner look) |
| Legend position | Format → Legend | Top or Right |

---

### Step 8: Publish & Schedule Refresh

1. **File** → **Publish** → select workspace
2. Go to **app.powerbi.com** → workspace → dataset
3. Click **Schedule refresh** → Settings:
   - Gateway: Personal gateway (for Snowflake)
   - Credentials: Snowflake username/password
   - Schedule: 4x daily (6AM, 12PM, 6PM, 12AM)
4. Done!

---

### Free Custom Visuals to Install (AppSource)

| Visual | Use For | How to Get |
|--------|---------|------------|
| **Charticulator** | Custom KPI cards | AppSource → search "Charticulator" |
| **Deneb** | Advanced Vega-Lite charts | AppSource → search "Deneb" |
| **Mapbox Visual** | Better maps than default | AppSource → search "Mapbox" |
| **Bullet Chart** | Target vs actual | AppSource → search "Bullet Chart" |
| **Infographic Designer** | Pictorial stats | AppSource → search "Infographic" |

To install: **Visualizations pane** → **...** → **Get more visuals** → search → Add

---

### Color Reference (Quick Copy)

```
Background:     #1B2838 (Dark Navy)
Card BG:        #2D3748 (Dark Gray)
Border:         #4A5568 (Medium Gray)
Text Primary:   #FFFFFF (White)
Text Secondary: #A0AEC0 (Light Gray)
Safe/Good:      #38B2AC (Teal)
Warning:        #F6AD55 (Amber)
Danger:         #FC8181 (Light Red)
Critical:       #E53E3E (Deep Red)
Accent:         #4299E1 (Electric Blue)
Purple:         #9F7AEA (Violet)
Green:          #48BB78 (Success)
```

---
## 22. Power BI Build Guide — Data Source & Chart Selection Per Page

### Rule:
- **Slicers needed?** → Use Fact + Dims (star schema)
- **No slicers (static)?** → Use Pre-aggregated views (faster)

---

### PAGE 0: Home (No Slicers — Use Pre-aggregated)

| Visual | Chart Type | Data Source | Fields |
|--------|-----------|-------------|--------|
| Total Frauds | **Card** | VW_PBI_EXEC_SUMMARY | TOTAL_FRAUDS |
| Fraud Rate | **Card** | VW_PBI_EXEC_SUMMARY | FRAUD_RATE_PCT |
| Total Loss | **Card** | VW_PBI_EXEC_SUMMARY | TOTAL_FRAUD_LOSS |
| Affected Customers | **Card** | VW_PBI_EXEC_SUMMARY | AFFECTED_CUSTOMERS |
| Monthly Trend | **Sparkline / Line (small)** | VW_PBI_MONTHLY_TREND | MONTH_NAME (X), FRAUD_RATE_PCT (Y) |
| Navigation Buttons | **Buttons** | None (Action → Page nav) | — |

---

### PAGE 1: Executive Summary (Slicers: Date, Region)

**Data Source:** Fact + Dims (for slicer filtering)

| Visual | Chart Type | Fields | Why This Chart |
|--------|-----------|--------|---------------|
| Total Transactions | **Card** | DAX: [Total Transactions] | Single big number |
| Fraud Count | **Card** | DAX: [Fraud Count] | Single big number |
| Fraud Rate % | **Card** | DAX: [Fraud Rate %] | Single big number |
| Total Fraud Loss | **Card** | DAX: [Total Fraud Loss] | Single big number |
| Monthly Trend | **Line Chart** | DIM_DATE[MONTH_NAME] (X), [Fraud Count] (Y), [Fraud Rate %] (secondary Y) | Shows direction over time |
| Top Categories | **Clustered Bar (horizontal)** | DIM_MERCHANT[CATEGORY] (Y), [Total Fraud Loss] (X) | Ranked comparison |
| Risk Distribution | **Donut Chart** | DIM_RISK_BAND[RISK_BAND] (legend), [Total Transactions] (values) | Part-of-whole |
| **Slicers** | **Dropdown / List** | DIM_DATE[MONTH_NAME], DIM_TIMEZONE[REGION] | Filter all visuals |

---

### PAGE 2: When (Slicers: Timezone, Amount Bucket)

**Data Source:** Fact + Dims

| Visual | Chart Type | Fields | Why This Chart |
|--------|-----------|--------|---------------|
| Fraud by Hour | **Column Chart** | DIM_TIME_SLOT[HOUR_OF_DAY] (X), [Fraud Rate %] (Y) | Distribution across 24h |
| Time of Day Split | **Stacked Bar** | DIM_TIME_SLOT[TIME_OF_DAY] (Y), [Fraud Count] (X), color by IS_BUSINESS_HOURS | Compare segments |
| Day of Week | **Bar Chart (horizontal)** | DIM_DATE[DAY_NAME] (Y), [Fraud Rate %] (X) | Ranked weekdays |
| Monthly Seasonality | **Area Chart** | DIM_DATE[MONTH_NAME] (X), [Fraud Rate %] (Y) | Trend with volume feel |
| Peak Hour Callout | **Card** | DAX: "5-7 PM = Peak" | Highlight insight |
| **Slicers** | **Dropdown** | DIM_TIMEZONE[REGION], DIM_AMOUNT_BUCKET[AMOUNT_BUCKET] | |

---

### PAGE 3: Where (Slicers: Risk Band, Category)

**Data Source:** Fact + Dims

| Visual | Chart Type | Fields | Why This Chart |
|--------|-----------|--------|---------------|
| US Map | **Filled Map** | DIM_LOCATION[STATE] (location), [Total Fraud Loss] (size), [Fraud Rate %] (color saturation) | Geographic pattern |
| Top 10 States by Loss | **Bar Chart (horizontal)** | DIM_LOCATION[STATE] (Y, Top 10), [Total Fraud Loss] (X) | Ranked impact |
| Top 10 States by Rate | **Bar Chart (horizontal)** | DIM_LOCATION[STATE] (Y, Top 10), [Fraud Rate %] (X) | Ranked risk |
| Region Pie | **Pie Chart** | DIM_TIMEZONE[REGION] (legend), [Fraud Count] (values) | Regional split |
| Zero-Fraud States | **Multi-row Card** | States with 0 fraud | Positive callout |
| **Slicers** | **Dropdown** | DIM_RISK_BAND[RISK_BAND], DIM_MERCHANT[CATEGORY] | |

---

### PAGE 4: Who (Slicers: Age Group, Gender)

**Data Source:** Fact + Dims

| Visual | Chart Type | Fields | Why This Chart |
|--------|-----------|--------|---------------|
| Age vs Fraud Rate | **Clustered Bar** | DIM_AGE_BUCKET[AGE_GROUP] (Y), [Fraud Rate %] (X), color by DIM_CUSTOMER[GENDER] | Compare age + gender |
| Gender Split | **Donut** | DIM_CUSTOMER[GENDER] (legend), [Fraud Count] (values) | M vs F ratio |
| Generation | **Stacked Column** | DIM_AGE_BUCKET[GENERATION] (X), [Fraud Count] (Y), color by GENDER | Generational view |
| Repeat Victims | **Table** | DIM_CUSTOMER[CUSTOMER_SK], [Fraud Count], [Total Fraud Loss] (Top N filter) | Detail drill |
| Heavy Target KPI | **Card** | DAX: count where fraud_count >= 10 | Highlight 118 targets |
| **Slicers** | **Dropdown** | DIM_AGE_BUCKET[AGE_GROUP], DIM_CUSTOMER[GENDER] | |

---

### PAGE 5: Why (Slicers: Risk Band, Amount Bucket)

**Data Source:** Fact + Dims

| Visual | Chart Type | Fields | Why This Chart |
|--------|-----------|--------|---------------|
| Risk Band Distribution | **Column Chart** | DIM_RISK_BAND[RISK_BAND] (X), [Total Transactions] (Y), [Fraud Rate %] (data labels) | Risk vs volume |
| Amount vs Risk Scatter | **Scatter Plot** | FACT[AMT] (X), FACT[RISK_SCORE] (Y), FACT[IS_FRAUD] (color) | Pattern detection |
| Score Correlation | **Table (formatted)** | Score Name, Correlation value (manual entry or measure) | Ranking predictors |
| Action Funnel | **Funnel Chart** | BLOCK → REVIEW → MONITOR → PASS (values: transaction count) | Operational flow |
| Threshold Table | **Matrix** | Threshold, Flagged, Caught, Precision | Decision support |
| **Slicers** | **Dropdown** | DIM_RISK_BAND[RISK_BAND], DIM_AMOUNT_BUCKET[AMOUNT_BUCKET] | |

---

### PAGE 6: How (Slicers: Category, Merchant Score)

**Data Source:** Fact + Dims

| Visual | Chart Type | Fields | Why This Chart |
|--------|-----------|--------|---------------|
| Category Loss | **Treemap** | DIM_MERCHANT[CATEGORY] (group), [Total Fraud Loss] (size) | Part-of-whole with hierarchy |
| Top Risky Merchants | **Table** | DIM_MERCHANT[MERCHANT], [CATEGORY], [MERCHANT_SCORE] (Top 10) | Detail list |
| Online vs POS | **100% Stacked Bar** | Online/POS (calculated), [Fraud Count] (values) | Channel comparison |
| Day-of-Month | **Line Chart** | DAX/calc: DAY(date) (X), [Fraud Rate %] (Y) | Salary day pattern |
| Recency Pattern | **Bar Chart (horizontal)** | Time bucket (Y), Fraud txns (X) | Gap analysis |
| Rapid-Fire KPI | **Card** | Count from FRAUD_ALERTS where type = 'RAPID_FIRE' | Alert volume |
| **Slicers** | **Dropdown** | DIM_MERCHANT[CATEGORY] | |

---

### PAGE 7: Alerts & Actions (Slicers: Alert Type, Severity, Date)

**Data Source:** FRAUD_ALERTS table (import separately, no star schema relationship)

| Visual | Chart Type | Fields | Why This Chart |
|--------|-----------|--------|---------------|
| Critical Alerts | **Card (Red)** | COUNT where SEVERITY = 'CRITICAL' | Urgency |
| High Alerts | **Card (Orange)** | COUNT where SEVERITY = 'HIGH' | Warning |
| Medium Alerts | **Card (Yellow)** | COUNT where SEVERITY = 'MEDIUM' | Monitor |
| Alert Table | **Table** | ALERT_TYPE, CC_NUM, MERCHANT, AMT, SEVERITY, CREATED_AT | Actionable detail |
| Alert Trend | **Line Chart** | DATE(CREATED_AT) (X), COUNT (Y), color by ALERT_TYPE | Volume over time |
| Unresolved Gauge | **Gauge** | COUNT where IS_RESOLVED = FALSE (vs total) | Backlog indicator |
| **Slicers** | **Dropdown** | ALERT_TYPE, ALERT_SEVERITY, DATE range | |

---

### PAGE 8: Pipeline Health (Slicers: Procedure, Date)

**Data Source:** PIPELINE_EXECUTION_LOG + PIPELINE_ERROR_LOG (import separately)

| Visual | Chart Type | Fields | Why This Chart |
|--------|-----------|--------|---------------|
| Pipeline Status | **Card (conditional)** | Latest EXECUTION_STATUS (Green/Red) | At-a-glance health |
| Duration Trend | **Line Chart** | BATCH_TIMESTAMP (X), DURATION_SECONDS (Y) | Performance trend |
| Success/Fail/Skip | **Stacked Column** | DATE (X), COUNT (Y), color by STATUS | Daily health |
| Avg Duration | **Card** | AVG(DURATION_SECONDS) | Benchmark |
| Rows Processed | **Area Chart** | BATCH_TIMESTAMP (X), ROWS_PROCESSED (Y) | Throughput trend |
| Error Table | **Table** | PROCEDURE_NAME, ERROR_MESSAGE, CREATED_AT | Debug info |
| **Slicers** | **Dropdown** | PROCEDURE_NAME, Date range | |

---

### Chart Selection Cheat Sheet

| When You Want To Show | Use This Chart |
|----------------------|----------------|
| Single number (KPI) | **Card** |
| Trend over time | **Line Chart** |
| Ranked comparison | **Bar Chart (horizontal)** |
| Distribution | **Column Chart (vertical)** |
| Part-of-whole | **Donut** or **Pie** |
| Geographic | **Filled Map** |
| Hierarchy + size | **Treemap** |
| Two variables pattern | **Scatter Plot** |
| Progress/target | **Gauge** |
| Funnel/flow | **Funnel Chart** |
| Detail/drill-through | **Table** or **Matrix** |
| Volume + trend | **Area Chart** |
| Segment comparison | **Stacked Bar/Column** |

---
## 23. Final Status — Power BI Dashboard Build

### Pages Completed ✅

| Page | Title | Status |
|------|-------|--------|
| 0 | Home (Command Center) | ✅ Built |
| 1 | Executive Summary (WHAT) | ✅ Built |
| 2 | When (WHEN) | ✅ Built |
| 3 | Where (WHERE) | ✅ Built |
| 4 | Who (WHO) | ✅ Built |

### Pages Remaining 🔨

| Page | Title | Status |
|------|-------|--------|
| 5 | Why (WHY) | ⏳ Pending |
| 6 | How (HOW) | ⏳ Pending |
| 7 | Alerts & Actions | ⏳ Pending |

---

### Final Tables in Power BI Model (13 total)

**Star Schema (10 — connected via relationships):**

| Table | Type | Used On |
|-------|------|--------|
| VW_PBI_FACT_TRANSACTIONS | Fact | All pages |
| VW_PBI_DIM_DATE | Dimension | Slicers, Page 1 |
| VW_PBI_DIM_CUSTOMER | Dimension | Page 4 |
| VW_PBI_DIM_MERCHANT | Dimension | Page 1, 6 |
| VW_PBI_DIM_LOCATION | Dimension | Page 3 |
| VW_PBI_DIM_TIME_SLOT | Dimension | Page 2 |
| VW_PBI_DIM_AMOUNT_BUCKET | Dimension | Slicers |
| VW_PBI_DIM_AGE_BUCKET | Dimension | Page 4 |
| VW_PBI_DIM_RISK_BAND | Dimension | Page 5 |
| VW_PBI_DIM_TIMEZONE | Dimension | Slicers |

**Standalone (3 — no relationships needed):**

| Table | Type | Used On |
|-------|------|--------|
| VW_PBI_EXEC_SUMMARY | Pre-aggregated | Page 0, 1 KPI cards |
| VW_PBI_MONTHLY_TREND | Pre-aggregated | Page 0 sparkline |
| VW_PBI_FRAUD_ALERTS | Standalone | Page 0 active alerts, Page 7 |

**Deleted from Power BI (5 — not needed):**
- ~~VW_PBI_FRAUD_BY_CATEGORY~~
- ~~VW_PBI_FRAUD_BY_HOUR~~
- ~~VW_PBI_FRAUD_BY_STATE~~
- ~~VW_PBI_FRAUD_BY_DEMOGRAPHICS~~
- ~~VW_PBI_RISK_DISTRIBUTION~~

---

### DAX Measures Created

| Measure | Formula |
|---------|--------|
| Fraud Count | `CALCULATE(COUNTROWS(FACT), FACT[IS_FRAUD] = 1)` |
| Total Transactions | `COUNTROWS(FACT)` |
| Fraud Rate % | `DIVIDE([Fraud Count], [Total Transactions], 0) * 100` |
| Total Fraud Loss | `CALCULATE(SUM(FACT[AMT]), FACT[IS_FRAUD] = 1)` |
| Avg Fraud Amount | `DIVIDE([Total Fraud Loss], [Fraud Count], 0)` |
| Affected Customers | `CALCULATE(DISTINCTCOUNT(FACT[CUSTOMER_SK]), FACT[IS_FRAUD] = 1)` |
| States with Fraud | `COUNTROWS(FILTER(VALUES(DIM_LOCATION[STATE]), CALCULATE(SUM(FACT[IS_FRAUD])) > 0))` |
| Heavy Targets | `COUNTROWS(FILTER(SUMMARIZE(FACT, FACT[CUSTOMER_SK], "FC", SUM(FACT[IS_FRAUD])), [FC] >= 10))` |
| Peak Hour Text | `"Peak: 5-7 PM"` |
| Top State Text | `"Top: NY"` |
| Top Age Text | `"Age: 55+"` |
| Top Score Text | `"Score: 0.19"` |
| Online Pct Text | `"Online: 44%"` |
| Active Alerts Text | `"Active: " & COUNTROWS(FILTER(VW_PBI_FRAUD_ALERTS, [IS_RESOLVED] = FALSE))` |

---

### Remaining Pages — Quick Build Guide

**Page 5: Why (Risk Factors)**
- Column Chart: DIM_RISK_BAND[RISK_BAND] (X) + [Fraud Rate %] (Y)
- Bar Chart: DIM_AMOUNT_BUCKET[AMOUNT_BUCKET] (Y) + [Fraud Rate %] (X)
- Text callout: Top predictors + threshold recommendation
- Slicers: DIM_RISK_BAND, DIM_AMOUNT_BUCKET

**Page 6: How (Patterns)**
- Treemap: DIM_MERCHANT[CATEGORY_BUCKET] (group) + [Total Fraud Loss] (size)
- Table: DIM_MERCHANT[MERCHANT], [CATEGORY], [MERCHANT_SCORE] (Top 10)
- Bar Chart: DIM_MERCHANT[CATEGORY_BUCKET] (Y) + [Fraud Count] (X)
- Slicers: DIM_MERCHANT[CATEGORY_BUCKET]

**Page 7: Alerts**
- 3 KPI Cards: Count by severity (CRITICAL, HIGH, MEDIUM) from VW_PBI_FRAUD_ALERTS
- Table: VW_PBI_FRAUD_ALERTS (all columns)
- Slicers: ALERT_TYPE, ALERT_SEVERITY

---

### Canvas & Theme
- Canvas: 1920 x 1080 (all pages)
- Theme: Fraud_Dark_Theme.json (applied)
- Home button: 200 x 70 on every page (top-left)

---
## 24. Final Project Summary & Portfolio Guide

---

### Project: Credit Card Fraud Detection & Analytics Pipeline

**Built by:** HARESHKUMAR99 
**Platform:** Snowflake + Power BI 
**Date:** May 2026

---

### What Was Built

| Component | Description |
|-----------|------------|
| **Medallion Architecture** | Bronze → Silver → Gold data pipeline |
| **Star Schema** | 1 Fact + 9 Dimension tables |
| **Data Enrichment** | Timezone, risk scores (velocity, location, amount, merchant), age, distance |
| **Real-Time Alerts** | Snowflake Alerts for rapid-fire fraud & high-risk merchants |
| **Stored Procedures** | Error handling, execution logging, schema checkpoint |
| **Streams & Tasks** | CDC pipeline with automated scheduling |
| **Power BI Dashboard** | 8-page interactive report with dark theme |
| **Cost Optimization** | Dedicated XS warehouse, auto-suspend, alert frequency tuning |

---

### Tech Stack

| Technology | Used For |
|-----------|----------|
| Snowflake | Data warehouse, streams, tasks, alerts, stored procedures |
| Power BI | Visualization, dashboards |
| SQL | All transformations, scoring, views |
| DAX | Measures, calculated fields |

---

### Key Results

| Metric | Value |
|--------|-------|
| Data processed | 555,719 transactions |
| Fraud detected | 2,145 cases (0.39%) |
| Total fraud loss | $1,133,325 |
| Risk scoring accuracy | 8.63% fraud rate in VERY HIGH band vs 0.09% in LOW |
| Pipeline cost | $0.36 credits for entire build |
| Dashboard pages | 8 interactive pages |
| Alert detection time | < 5 minutes |

---

### Dashboard Pages

| Page | Title | Question Answered |
|------|-------|------------------|
| 0 | Command Center | Navigation + Quick Stats |
| 1 | Executive Summary | WHAT is happening? |
| 2 | When | WHEN does fraud peak? |
| 3 | Where | WHERE are the hotspots? |
| 4 | Who | WHO is being targeted? |
| 5 | Why | WHY did fraud succeed? |
| 6 | How | HOW do fraudsters operate? |
| 7 | Alerts & Actions | What needs attention? |

---

### GitHub Repository Structure

```
credit-card-fraud-analysis/
├── README.md
├── screenshots/
│   ├── page0_home.png
│   ├── page1_executive.png
│   ├── page2_when.png
│   ├── page3_where.png
│   ├── page4_who.png
│   ├── page5_why.png
│   ├── page6_how.png
│   └── page7_alerts.png
├── sql/
│   ├── 01_create_schemas.sql
│   ├── 02_bronze_layer.sql
│   ├── 03_silver_layer.sql
│   ├── 04_gold_dimensions.sql
│   ├── 05_gold_fact.sql
│   ├── 06_stored_procedures.sql
│   ├── 07_streams_tasks.sql
│   ├── 08_alerts.sql
│   ├── 09_views_powerbi.sql
│   └── 10_cost_optimization.sql
├── powerbi/
│   ├── Fraud_Dark_Theme.json
│   └── Fraud_Analysis.pbix
├── docs/
│   ├── architecture_diagram.png
│   ├── data_dictionary.md
│   └── cost_analysis.md
└── notebook/
    └── Fraud_Analysis.ipynb
```

---

### README.md Template

```markdown
# Credit Card Fraud Detection & Analytics Pipeline

## Overview
End-to-end fraud detection pipeline built on Snowflake with medallion architecture,
real-time alerting, and an 8-page Power BI dashboard.

## Architecture
![Architecture](docs/architecture_diagram.png)

### Data Flow
```
Source → Bronze (raw) → Silver (enriched + scored) → Gold (star schema) → Power BI
```

## Key Features
- **Medallion Architecture:** Bronze/Silver/Gold with zero data loss
- **Risk Scoring:** 4 custom scores (velocity, location, amount, merchant)
- **Real-Time Alerts:** Snowflake Alerts for rapid-fire & high-risk merchants
- **Star Schema:** 1 Fact + 9 Dimensions optimized for Power BI
- **Schema Evolution:** Hybrid VARIANT approach for future-proofing
- **Cost Optimized:** $0.36 total compute for 555K row pipeline build

## Results
| Metric | Value |
|--------|-------|
| Fraud Rate | 0.39% |
| Total Loss Identified | $1.13M |
| Top Predictor | Amount Score (0.19 correlation) |
| Recommended Threshold | 0.6 (catches 30% fraud, 8% precision) |
| Peak Fraud Time | 5-7 PM local time |
| Highest Loss State | NY ($99K) |

## Dashboard Preview
![Dashboard](screenshots/page1_executive.png)

## Tech Stack
- Snowflake (warehouse, streams, tasks, alerts, procedures)
- Power BI (visualization)
- SQL & DAX

## How to Run
1. Execute SQL scripts in order (01-10)
2. Import `Fraud_Dark_Theme.json` in Power BI
3. Connect Power BI to Snowflake GOLD schema
4. Load VW_PBI_* views

## Cost
- Build cost: $1.08
- Daily operational: $0.21/day
- Monthly at 1M/day: ~$35/month
```

---

### Where to Publish Your Portfolio

| Platform | Best For | How |
|----------|---------|-----|
| **GitHub** | Code + SQL + documentation | Push repo with README + screenshots |
| **LinkedIn** | Professional visibility | Post with dashboard screenshots + summary |
| **Medium / Hashnode** | Technical blog | Write article explaining your approach |
| **Tableau Public** | Visual portfolio (alternative to Power BI) | Rebuild 2-3 key pages in Tableau Public |
| **Personal website** | All-in-one showcase | GitHub Pages (free) with project gallery |
| **NovyPro** | Power BI specific portfolio | Upload .pbix file — interactive online |

---

### Recommended Portfolio Strategy

**Step 1: GitHub**
1. Create repo: `credit-card-fraud-analysis`
2. Add all SQL files (organized in folders)
3. Add screenshots of each dashboard page
4. Write README with architecture, results, tech stack

**Step 2: LinkedIn Post**
```
🔍 Built an end-to-end Fraud Detection Pipeline on Snowflake

✅ Medallion Architecture (Bronze → Silver → Gold)
✅ Custom Risk Scoring (4 algorithms)
✅ Real-time Alerts (< 5 min detection)
✅ 8-page Power BI Dashboard
✅ Cost: $0.36 to process 555K transactions

Key finding: Amount deviation is the #1 fraud predictor — 
not time of day, not location.

Tech: Snowflake | Power BI | SQL | DAX

#DataEngineering #Snowflake #FraudDetection #PowerBI
```

**Step 3: NovyPro (Power BI Portfolio)**
1. Go to `novypro.com` → Create free account
2. Upload your `.pbix` file
3. It becomes an interactive online dashboard anyone can click
4. Share link on LinkedIn/resume

---

### Interview Talking Points

| Question | Your Answer |
|----------|------------|
| What architecture did you use? | Medallion (Bronze/Silver/Gold) with star schema in Gold |
| Why not just one table? | Data loss risk, no enrichment layer, can't audit lineage |
| How do you handle schema changes? | Hybrid VARIANT in Bronze + ALERT_SCHEMA_DRIFT |
| How do you detect fraud real-time? | Snowflake Alerts check every 5 min for rapid-fire & high-risk patterns |
| How did you optimize cost? | Dedicated XS warehouse, auto-suspend 60s, reduced alert frequency |
| What's the #1 fraud predictor? | Transaction amount (0.19 correlation) — not time or location |
| How does Power BI refresh? | Import mode from pre-aggregated views (5-50 rows each) + star schema for drill-through |

---
## 25. Phase 2 — Deep-Dive Analysis (Ready to Execute)

SQL file: `/sql/09_deep_dive_analysis.sql`

| # | Analysis | What It Reveals | Key Question |
|---|----------|----------------|--------------|
| A | Fraud Chains | How long fraudsters exploit one card | "How many hours between first and last fraud on same card?" |
| B | First Fraud Timing | Time from first legit txn to first fraud | "How soon after compromise does fraud start?" |
| C | Merchant Patterns | Which merchants are hit together within 24h | "Are there fraud rings sharing stolen cards?" |
| D | City Clustering | Fraud per 100K population | "Which small cities have disproportionate fraud?" |
| E | Weekend vs Weekday | Behavioral shifts by category + time | "Do fraudsters behave differently on weekends?" |
| F | Amount Distribution | Exact dollar cutoff where fraud spikes | "At what exact amount does fraud probability jump?" |
| G | Customer Lifetime Value | High-value customers losing most | "Which VIP customers should we protect first?" |
| H | Repeat Merchant Fraud | Same merchant, multiple different victims | "Which merchants are compromised?" |

### How to Use
- Run each query individually in Snowflake
- Results can be added as new Power BI pages or exported to Excel
- These go deeper than the current dashboard into investigative analysis

---

## 26. Data Governance Updates

SQL file: `/sql/08_governance.sql`

| Protection | Applied To | What It Does |
|-----------|-----------|-------------|
| Masking Policy (String) | CC_NUM, FIRST, LAST, GENDER, STREET | Shows `***MASKED***` for non-admin roles |
| Masking Policy (Number) | ZIP | Shows `0` for non-admin roles |
| Masking Policy (Float) | LAT, LONG | Shows `0.0` for non-admin roles |
| Masking Policy (Date) | DOB | Shows `1900-01-01` for non-admin roles |
| CC_NUM in Power BI views | VW_PBI_DIM_CUSTOMER | Shows `4469****8234` (masked in view) |
| CC_NUM in Alerts view | VW_PBI_FRAUD_ALERTS | Shows `4469****8234` (masked in view) |
| JOB_BUCKET added | DIM_CUSTOMER | 478 jobs → 12 logical buckets |
| CATEGORY_BUCKET added | DIM_MERCHANT | 14 categories → 5 channel buckets |

---

## 27. Complete File List (Download All)

| File | Purpose |
|------|---------|
| `/Fraud_Analysis.ipynb` | Main notebook (all analysis + notes) |
| `/Fraud_Dark_Theme.json` | Power BI theme file |
| `/sql/01_create_schemas.sql` | Create BRONZE, SILVER, GOLD schemas |
| `/sql/02_bronze_layer.sql` | Raw data + VARIANT + monitoring tables |
| `/sql/03_silver_layer.sql` | Enriched data + all scoring |
| `/sql/04_gold_star_schema.sql` | 9 dimensions + 1 fact table |
| `/sql/05_stored_procedures.sql` | All 4 stored procedures |
| `/sql/06_streams_tasks_alerts.sql` | Streams, tasks, alerts, privileges |
| `/sql/07_views_powerbi.sql` | All Power BI views (masked) |
| `/sql/08_governance.sql` | Masking policies for PII |
| `/sql/09_deep_dive_analysis.sql` | Phase 2 investigative queries |

**Total: 11 files**